<a href="https://colab.research.google.com/github/IanGYan/DelegationDAO2/blob/main/%E5%A4%A7%E6%A8%A1%E5%9E%8B%E6%98%AF%E5%A6%82%E4%BD%95%E7%82%BC%E6%88%90%E7%9A%84%EF%BC%88%E8%AF%BE%E4%BB%B6%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **大语言模型 (LLM) 是如何炼成的？—— 从三个维度看演进之路**

讲师：何亮（博士，南京大学人工智能学院）

课时：3 课时

大型语言模型（LLMs）如 GPT、Claude、Gemini、DeepSeek 等在近年来取得了令人瞩目的成就，它们能够理解和生成极其自然、连贯且富有信息的文本。但这并非一蹴而就，<font color="red">**LLM 的诞生是自然语言处理（NLP）领域数十年技术积累和演进的结果**</font>。

本课件将从三个核心维度，带你回顾这段波澜壮阔的技术发展史：
1.  **表示的革命和大一统 (Representation Evolution):** 如何将文字、甚至世界万物表示为计算机能够理解和处理的形式？
2.  **架构的革命和大一统 (Architecture Evolution):** 用什么样的模型结构来捕捉语言乃至知识中的复杂模式？
3.  **范式的革命和大一统 (Paradigm Evolution):** 如何训练和使用这些模型来完成各种任务？

通过理解这些演进脉络，我们将能更深刻地认识 LLM 的能力来源、内在机理以及未来的发展方向。

## 0. 引言

![history.png](https://drive.google.com/uc?id=1J4w0e1stU3q3xu9pVvkjO0K882Iw5MBv)

LLM的发展是一个不断迭代、融合的过程。这三条脉络相互交织、共同推动了LLM的进步：
- **表示的进化提供了更有效的输入和特征**：计算机表示信息从早期无法捕捉语义、忽略语序的离散表示（如One-hot），发展到能体现词语相似性的静态分布式表示（如Word2Vec），但后者无法处理一词多义。关键突破是上下文相关表示（如ELMo、Transformer Embeddings），它们根据语境动态生成词向量，解决了上下文依赖问题。随后，多模态表示（如CLIP）将文本、图像等信息映射到统一空间，实现跨模态理解。LLM正是建立在强大的上下文和多模态表示基础上，以理解和生成丰富、连贯的内容。
- **架构的进化提供了更强大的建模能力**：AI架构从难以处理序列的早期网络（ANN/MLP），演进到能处理序列但受限于长距离依赖和串行计算的RNN（含LSTM/GRU）。注意力机制缓解了长距离问题。革命性的Transformer架构完全基于自注意力，实现了并行计算和强大的长距离依赖捕捉，成为训练大型模型的基础。BERT（Encoder）和GPT（Decoder）等是其关键变体。LLM的强大能力很大程度上源于Transformer带来的高效训练和序列建模优势。
- **范式的进化使得 LLM 更加灵活和通用**：AI任务解决范式从依赖专家规则、泛化差的系统，转向从数据中学习的统计机器学习（依赖特征工程），再到自动学习特征的深度学习（需大量标注数据和定制模型）。关键转变为预训练/微调范式，通过大型通用模型加少量数据微调实现知识迁移和模型复用。最终，提示学习/上下文学习范式实现了无需微调，通过自然语言提示即可引导同一大模型完成多样任务（零/少样本），极大促进了任务的统一，展现了通用智能的潜力。

## 1. 表示的革命和大一统——万物皆可 Embedding
自然语言处理的首要问题是如何让计算机“理解”文字。这里的“理解”第一步就是如何表示（Represent）文本。表示方法的好坏，直接决定了模型能力的上限。

### 1.1 离散表示：独热编码 (One-hot Encoding) 与 词袋模型 (Bag-of-Words, BOW)

**One-hot Encoding:** 最早期的词表示方法之一。假设词典中有 V 个词，每个词被表示为一个 V 维的向量，其中该词对应的维度为 1，其余维度均为 0。

**Bag-of-Words:** 将文档（句子或段落）表示为一个向量。向量的维度也是词典大小 V。每个维度代表一个词，其值可以是该词在文档中出现的次数（词频，TF）或是否出现（二进制）。它忽略了词序和语法。

> **⚠️ 问题**
> * **高维稀疏 (High-dimensionality & Sparsity):** 词典很大时，向量维度非常高，且大部分元素为 0，存储和计算效率低。
> * **无法表示语义相似度 (Semantic Similarity):** One-hot 向量之间都是正交的（点积为0），无法体现 "猫" 和 "狗" 比 "猫" 和 "苹果" 更相似。
> * **词袋模型丢失语序:** "我不喜欢你" 和 "你喜欢我" 的 BOW 表示可能非常相似，但含义相反。

### 1.2 分布式表示 (Distributed Representation): Word2Vec / GloVe

🤔 有谁知道下面这些词的含义吗？
    
> <font color="red" size="8">  縕袍 (yùn páo)  | 逡巡 (qūn xún) |   哂纳 (shěn nà) <br>
     滥觞 (làn shāng) | 龃龉 (jǔ yǔ) | 怊怅 (chāo chàng) </font>

如果这样呢？
    
> 数九寒天，那穷苦的书生家徒四壁，身上只穿着一件<font color="red">**縕袍**</font>，在刺骨的寒风中瑟瑟发抖，连笔都快握不住了。\
> —— 用旧絮填充的袍子，通常是贫苦人家穿的。
    
> 接到那个棘手的任务，他心里没底，在办公室门口<font color="red">**逡巡**</font>了半天，几次想推门进去汇报，又都缩回了手。\
> —— 形容因为犹豫、顾虑而徘徊不前、想做又不敢做的样子。
    
> 区区薄礼，不成敬意，万望先生<font color="red">**哂纳**</font>。\
> —— 客气的、带有谦逊意味的请人收下礼物的说法。
    
> 这条波涛汹涌的大河，其<font color="red">**滥觞**</font>之处，不过是雪山融水汇成的一股涓涓细流。\
> —— 江河的发源地（水刚流出时浅得只能浮起酒杯），引申为事物的起源或开端。
    
> 由于在核心问题上长期存在<font color="red">**龃龉**</font>，两部门之间的合作一直磕磕绊绊，难以顺畅推进。\
> —— 上下牙齿对不齐，引申为意见不合、互相抵触、关系不和睦。
    
> 得知盼望已久的好友最终未能赴约，他独自站在窗前，望着淅沥的雨丝，心中不禁涌起一阵<font color="red">**怊怅**</font>，默默地叹了口气。\
> —— 一种因失意或失望而引起的伤感、惆怅、若有所失的情绪。

分布式表示的核心思想是“词的含义由其上下文决定”（Distributional Hypothesis）。不再使用稀疏的高维向量，而是将每个词映射到一个 **低维、稠密** 的向量（称为词嵌入，Word Embedding）。这个向量是通过神经网络在大量文本上训练得到的，使得**语义或语法上相似的词，其向量在向量空间中也更接近**。

**Tokenization**

进行分布式表示学习时，通常不会直接以<strong>词</strong>为单位，而是把文字转为 <strong>Token</strong>，这个过程叫做 Tokenization。Token 是一个抽象的概念，可以是一个 subword、一个单词、一个字、一个词组等。以下是 DeepSeek R1 的词汇表：
![TOKEN](https://drive.google.com/uc?id=1VHL4ahVDLJlMuorADaOc4w7a2N2CvOqh)


> **🤔思考一下：为什么不直接用词，而要费这劲搞什么 Token ？**
>  * **控制住了“字典”大小**：通过把词拆分成更小的单元（Token），我们只需要记住有限数量的 Token（比如几万个，而不是几百万个词）和它们的 embedding 就够了。
>  * **处理生词**：当遇到一个生词时，即使整个词不在 Token 字典里，但很可能组成它的字或片段在字典里。模型可以通过组合这些已知 Token 的 embedding，来推断出这个生词的大致含义或用法，而不是完全卡壳。
>  * **间接利用了词的内部结构**：通过把词拆分成有意义的子部分（比如词根、前后缀对应的 Token），模型在学习这些 Token 的 embedding 时，就能捕捉到它们携带的语义信息。比如，模型能学到带有 "un-" 或 “没-” 的 Token 通常表示否定。这样，即使没见过 “unhelpful”，但见过 “un-” 和 “helpful” 对应的 Token，也能理解其含义。

**Word2Vec (Mikolov et al., 2013)：** 包含两种主要的训练架构：
  - **CBOW (Continuous Bag-of-Words):** 根据上下文词预测中心词。
     1.  **训练快：** 因为每次是把周围词的信息“平均”一下，去预测一个词，计算量相对小。
     2.  **对高频词效果好：** 常见词的周围环境（上下文）出现次数多，模型容易学习它们的“平均”特征。
  - **Skip-gram:** 根据中心词预测上下文词。
     1.  **对低频词效果更好：** 因为是拿一个中心词去预测多个周围词，即使中心词不常见，它与周围具体词的关系也能被单独学习，能学到更精细的表达。
     2.  **在小型数据上表现好：** 少量数据也能学习到词与词之间的具体关系。

在实际应用中，Skip-gram 因为其在捕捉罕见词和细微语义关系上的优势，往往更受欢迎一些，尽管训练会慢一些。

![Word2Vec](https://drive.google.com/uc?id=1MHEHVChEhVk8Bhq6yQoDoQhzR0SkSS1q)


**GloVe (Pennington et al., 2014)：** 是一种基于全局词共现统计的无监督学习算法，通过优化损失函数生成稠密词向量，以捕捉词语之间的语义和语法关系。

![GloVe](https://drive.google.com/uc?id=1MF1NMfFM9Fnq27wdIe4hmnTgWlJXnGZc)

<div class="alert alert-info">
<strong>简单打个比方：Word2Vec靠猜邻居（局部预测），GloVe靠算总账（全局统计+局部关联）。</strong>
</div>

**优势**
1.  **低维稠密:** 向量维度显著降低（通常 50-300 维），稠密表示更利于计算和存储。
2.  **蕴含语义信息:** 通过训练，向量可以捕捉到词之间的语义关系。例如 `vector('国王') - vector('男人') + vector('女人')` 在向量空间中接近 `vector('王后')`。

![embedding.webp](https://drive.google.com/uc?id=1M0a2ijPdAUWqmtbxWR_2l6-l62hV3TSn)

> **⚠️ Word2Vec/GloVe 的局限性：**\
> 尽管分布式表示取得了巨大成功，但它们仍然存在一个核心问题：**无法处理一词多义 (Polysemy)**。每个词只有一个固定的向量表示，无法根据上下文动态调整。例如，"bank" 在 "river bank"（河岸）和 "bank account"（银行账户）中的含义完全不同，但它们的 Word2Vec 向量是相同的。这限制了模型对语境的理解深度。

[Word Embedding 可视化演示](https://projector.tensorflow.org/)

### 1.3 上下文相关表示 (Contextualized Representation)

为了解决 Word2Vec 等静态词向量无法处理“一词多义”的问题，研究者们提出了上下文相关的词表示方法。核心思想是：<font color="red">**一个词（或子词 Token）的表示应该根据它所处的具体语境动态生成**</font>。

**ELMo (Embeddings from Language Models, Peters et al., 2018):** 使用多层双向 LSTM (biLSTM) 训练双向语言模型（独立训练前向与后向 LSTM 并拼接）。一个词的最终表示是其底层输入向量与 biLSTM 各层输出向量的加权组合。由于前、后向 LSTM 的隐状态分别依赖于之前的上文序列和之后的下文序列，因此得到的表示是上下文相关的。它是“深层”的，因为融合了多层 LSTM 不同抽象层级的语境信息（如底层偏向句法、高层偏向语义）。

**基于 Transformer 的动态表示 (Vaswani et al., 2017; Devlin et al., 2019 - BERT; Radford et al., 2018/2019 - GPT/GPT-2):** 基于 Transformer 架构（我们将在角度二详细讨论）。通过其核心机制——**自注意力 (Self-Attention)**，模型在计算一个词的表示时，能够直接“关注”序列中相关位置的词（全句所有词或所有前序词），并根据相关性加权聚合它们的信息。这使得模型彻底摆脱了 RNN 的序列顺序计算限制，能够非常高效地捕捉长距离依赖和复杂的上下文关系。
- **BERT (Bidirectional Encoder Representations from Transformers):** 采用 Transformer Encoder，通过掩码语言模型 (MLM) 和下一句预测 (NSP) 任务进行预训练，每一层都能同时融合左右两侧的上下文（深层双向）。
- **GPT (Generative Pre-trained Transformer):** 采用 Transformer Decoder，通过标准的从左到右自回归语言模型任务进行预训练（利用掩码自注意力仅关注上文），更擅长文本生成任务（单向/自回归）。

![contextual](https://drive.google.com/uc?id=1i2-p3hAtCbdwcaS4uC2i8AxQnnNllGXD)

[上下文相关 Embedding 示意](https://drive.google.com/uc?id=1Jz2GVxKa5YBkbTsReHIGKSYx3erd0ezf)

**优势**
1. **动态表示:** 同一个词在不同语境下会产生不同的向量表示。例如，"bank" 在 "river bank"（河岸）和 "bank account"（银行账户）中的向量会截然不同。
2. **更强的上下文理解:** 通过深层序列模型（ELMo 的多层 biLSTM）或并行自注意力机制（Transformer），模型能捕捉更丰富、更长距离的语法与语义信息，显著提升了下游任务（如问答、情感分析、机器翻译）的性能。

### 1.4 多模态融合表示 (Multimodal Representation): CLIP
世界是多模态的，信息不仅仅来自文本，还来自图像、声音、视频等。ELMo/BERT/GPT等纯文本模型无法处理和理解文本以外的信息（如图像、声音），限制了模型在现实世界应用中的能力。为了让模型具备更全面的理解能力，研究者开始探索如何将不同模态的信息融合到统一的表示空间中。

**CLIP (Contrastive Language-Image Pre-training, Radford et al., 2021):** 通过对比学习，训练一个图像编码器和一个文本编码器。目标是让匹配的（图像，文本）对在表示空间中尽可能接近，而不匹配的对尽可能远离。这使得 CLIP 能够理解图像和文本之间的关系，实现零样本图像分类、图像文本检索等。

![clip](https://drive.google.com/uc?id=1oU14y_4TqjxXosgv23mUeURxJXS3BDMN)

**优势：**
1.  **跨模态理解:** 模型能够关联和理解不同模态（如图像和文本）的信息。
2.  **更广泛的应用:** 支持图文生成、视觉问答 (VQA)、图像描述、多模态对话、机器人控制等新任务。
3.  **更强的泛化能力:** 通过多模态学习，模型可能获得对世界更全面的“常识”理解。

### 1.5 应用时为什么也要理解 Tokenization 和 Embedding？

每个预训练语言模型，都有自己配套的切词词表（Tokenizer）和独立训练出的“语义理解坐标系”（Embedding 空间）。哪怕两个模型输出的向量维度完全相同，它们各自坐标轴的方向也截然不同。因此，**建库（存文档）用的 Embedding 模型，和检索（搜问题）用的 Embedding 模型，得是同一个模型！**

### 1.6 前沿进展

#### **进展一：全感官大一统（Omni-Modal 与柏拉图假说）**

* **过去**：管文字的 AI、管图片的 AI、管声音的 AI 各画各的地图，彼此语言不通。
* **现在（Omni-Modal 全模态表征）**：最新的多模态大模型把**文字、图片、视频、音频甚至手机屏幕点击动作**，全部映射进同一个共享的坐标系里。

##### **柏拉图表征假说（The Platonic Representation Hypothesis）**

刚才的实验里我们看到：国产中文模型和谷歌英文模型的向量直接点积只有 `0.0639`（完全不兼容）。但这是否意味着不同大模型眼中的世界结构是彻底鸡同鸭讲的呢？

2024 年，MIT 团队在 ICML 发表了轰动学界的论文 **[《The Platonic Representation Hypothesis》](https://arxiv.org/abs/2405.07987)**，揭示了一个奇妙的真相：
* 不同模型之所以不能直接点积，只是因为它们的**坐标系朝向**不同（好比一个人以北京为原点、上北下南画地图；另一个人以纽约为原点、左北右南画地图，直接拿坐标相乘当然不对）。
* 但如果你看模型内部**概念与概念之间的相对距离关系**（比如地图上“北京到上海的距离”与“北京到广州的距离”之比），随着模型越来越大、越来越聪明，**无论是中文模型、英文模型，甚至是纯看图片的视觉模型，它们内部的语义几何结构正在惊人地收敛到同一个唯一的底层形态**！
* 就像柏拉图《理想国》里的“洞穴寓言”：文字、图像、声音都是真实客观世界投射在墙上的影子，当大模型足够强大时，它们透过各自的影子，都在殊途同归地逼近同一个**客观物理世界的真实表征**。

最新的数学定理（JEPA 正交同构定理[《Proving and Improving the Platonic Representation Hypothesis with JEPAs》](https://arxiv.org/abs/2610.39302)）证明：这两个独立训练的空间，仅仅相差了一个**旋转角度（正交旋转变换）**——就像两个人从不同角度拍同一座雕塑，把照片转个方向就能严丝合缝地重合！

#### **进展二：具身智能与 JEPA 世界模型**

当 AI 不再只是陪你聊天的对话框，而是装进机器人身体里去倒茶、叠衣服、开汽车时，它需要的 Embedding 就是能预测下一秒物理变化的世界模型。

##### **为什么传统的“像素级预测”在机器人身上会翻车？**
以前科学家让 AI 看海量人类做家务的视频，要求它**预测下一秒画面的每一个像素（Pixel Reconstruction）**。结果遭遇了严重的滑铁卢：
* **新手数树叶困境（外观噪声偏置）**：就像新手开车时紧盯路边每一片被风吹动的树叶、挡风玻璃上的反光，大脑瞬间被无关的细节撑爆；而老司机开车只看**前车距离、车道线、自身车速**这几个核心状态变量。
* **抄答案作弊陷阱（信息泄露坍缩）**：如果同时给 AI 看“推杯子前的画面”和“推杯子后的画面”让它总结动作，AI 会偷懒直接把后一张画面的像素抄下来，根本没有学会“用力推 $\rightarrow$ 杯子往右滑”的物理因果律。

##### **最新突破：VLA-JEPA（在抽象概念空间里“脑补”未来）**
图灵奖得主 Yann LeCun 提出的 **JEPA（联合嵌入预测架构）** 及其最新的机器人版本 **VLA-JEPA** 彻底改变了做法：
1. **扔掉无关像素**：先把画面压缩成一个干净的核心状态点（过滤掉背景灯光闪烁、窗帘晃动）；
2. **严格因果脑补**：在纯抽象的 Embedding 空间里，根据当前的动作指令（比如“向右推”），直接预测下一秒状态点会移动到哪个坐标！



#### **进展三：符号-神经混合 Box Embedding**

##### **为什么传统的“点坐标”不会做逻辑推理？**
在传统的 Embedding 空间里，每个词都是地图上的**一个点**：
* 假设 `【柯基】` 是一个点，`【狗】` 也是一个点，它们俩相距 1 厘米。
* **问题来了**：只看这两个点的距离，计算机能分清是**所有柯基都是狗**，还是**所有狗都是柯基**吗？
* **不能！** 因为两点之间的距离是**对称的**（A 到 B 的距离永远等于 B 到 A 的距离），但现实世界里的逻辑包含关系是**绝对不对称的**！

##### **Box Embedding（把概念变成有面积的“矩形领地”）**
为了教会神经网络严密的符号逻辑，科学家不再把概念表示成一个没有大小的“点”，而是表示成一个**矩形盒子（Box Embedding / 超矩形）**：
1. **上下级包含关系（是谁的子类）** $\rightarrow$ **大盒子套小盒子**！`【柯基】` 的小盒子完全装在 `【狗】` 的大盒子里，一眼就能看出谁包含谁。
2. **逻辑“且”运算（交集推理）** $\rightarrow$ **两个盒子重叠的地方**！比如 `【哺乳动物】` 的盒子和 `【会飞的动物】` 的盒子十字相交，重叠出来的那块区域是什么？正是 **`【蝙蝠】`**！
3. **逻辑互斥（绝不可能同时发生）** $\rightarrow$ **两个盒子完全不挨着**！

##### **从“画硬盒子”进化到“大模型逻辑护栏”**
早期的硬盒子有两个弱点：一是两个盒子不挨着时容易算不出优化方向；二是“盒子外面的区域”不再是个盒子，很难表达“不是哺乳动物（逻辑否定）”。
到了 2025–2026 年，科学家对它进行了三大升级：
1. **高斯软盒子（如 WWW 2026 的 TaxoBell）**：把硬邦邦的矩形边界变成像“云团”一样带有概率渐变的高斯盒子，既能算面积包含，又能衡量 AI 的不确定性；
2. **更丰富的几何形状（圆锥、代数格与 GeometrE）**：引入扇形圆锥和代数格，连“非 A”这种否定逻辑也能严密计算；
3. **给大模型当“逻辑探针”（如 2026 年的 Prompt2Box 与 NeurOWL）**：直接把 ChatGPT 等大模型内部的隐状态映射成几何盒子，用来自动检查大模型有没有犯“指鹿为马、上下级颠倒”的逻辑幻觉，甚至从杂乱数据里自动长出一棵知识树！


### ✅ 小结：表示的演进回顾

|年代 | 阶段 | 表示形式 | 主要特点 |
|---------|------|-----------|----------|
|1950s | One-hot/BOW | 稀疏向量 | 无语义，无上下文 |
|2013 | Word2Vec | 稠密词向量 | 语义相似性，可视化 |
|2018 | BERT / ELMo | 上下文相关向量 | 多义词处理，更强表达 |
|2021 | CLIP | 多模态嵌入 | 图文对齐，语义共享 |
|2023～ | Embodied / Symbolic Hybrid | 可组合、可解释 | 面向智能体、推理能力 |

> 🌟 我们可以看到，表示方法经历了从 <strong><font color="red">离散、稀疏、静态</font></strong> 到 <strong><font color="red">稠密、动态、上下文相关</font></strong>，再到 <strong><font color="red">多模态融合</font></strong>，并正在向 <strong><font color="red">与物理世界交互 (Embodied)</font></strong> 和 <strong><font color="red">与符号逻辑结合 (Neuro-Symbolic)</font></strong> 的方向发展。每一次演进都旨在让计算机更深入、更全面地理解和表示信息，为更强大的 AI 能力奠定基础。

## 2. 架构的革命和大一统 —— 从序列处理到全局关联

有了表示方法，我们还需要强大的模型 **架构 (Architecture)** 来学习数据中的复杂模式，尤其是语言中存在的序列依赖关系和长距离关联。模型架构的演进，是 LLM 能力提升的另一个关键驱动力。

### 2.1 人工神经网络 (ANN / Multi-Layer Perceptron, MLP)

#### 2.1.1 什么是神经网络？

神经网络 (Neuron Network, NN) 是由大量神经元组成的网络，每个神经元都有输入、输出、权重和激活函数。一个典型的神经网络结构如下图所示：

一个神经元：$y=f(\sum_i w_i\cdot x_i + b)$

![neuron](https://drive.google.com/uc?id=17wdcWrOXBzvvGlhXB9xhMNqKwCthxTb7)

图源：Sukanya Bag

把很多神经元连接起来，就成了神经网络：$y=f(\sum_i w_i\cdot x_i + b)$、$z=f(\sum_i w'_i\cdot y_i + b')$、$\tau=f(\sum_i w''_i\cdot z_i + b'')$、...

![network](https://drive.google.com/uc?id=1IDcyZbI7M8mZaLDFZ2lHQrP7ysIHO1Fs)

最基础的神经网络结构，由输入层、一个或多个隐藏层和输出层组成。层与层之间通过权重连接，通过激活函数引入非线性。信息单向（前向）传播。
- 输入层：接收输入信号
- 隐藏层：处理输入信号
- 输出层：输出结果
- 权重：调节输入信号的重要性
- 激活函数：决定神经元是否激活
- 损失函数：评估模型预测结果与真实结果的差异

这里的$f$叫激活函数（Activation Function）。在神经网络中，激活函数负责向网络中引入非线性因素。**如果没有激活函数，无论神经网络有多少层，最终都等同于一个简单的线性变换，无法学习复杂的数据模式。** 激活函数作用于每个神经元的输出上。



激活函数有很多种形式，下表对比了常用的激活函数的主要特性、优点和缺点：

| 特性/函数        | Sigmoid (Logistic)                      | Tanh (双曲正切)                         | ReLU (修正线性单元)                   | Leaky ReLU                            | PReLU (参数化ReLU)                   | ELU (指数线性单元)                     | Softmax                               |
| :--------------- | :-------------------------------------- | :-------------------------------------- | :------------------------------------ | :------------------------------------ | :----------------------------------- | :------------------------------------- | :------------------------------------ |
| **数学公式**     | `σ(x) = 1 / (1 + e^(-x))`              | `tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))` | `f(x) = max(0, x)`                    | `f(x) = max(αx, x)` (α通常很小, 如0.01) | `f(x) = max(αx, x)` (α是可学习参数)   | `f(x) = x if x>0` <br> `α(e^x - 1) if x≤0` | `σ(z)_i = e^(z_i) / Σ(e^(z_j))`       |
| **函数图像**     | S形曲线                                 | S形曲线 (比Sigmoid陡峭)               | 折线 (x>0时为y=x, x≤0时为y=0)          | 折线 (x<0时有微小正斜率)              | 折线 (x<0时斜率可学习)               | x>0时为y=x, x≤0时为平滑负指数曲线      | (作用于向量, 输出概率分布)            |
| **输出范围**     | (0, 1)                                  | (-1, 1)                                 | [0, +∞)                               | (-∞, +∞)                              | (-∞, +∞)                             | (-α, +∞)                             | (0, 1) 且各项和为1                     |
| **是否零中心**   | 否 (输出恒正)                           | 是 (输出均值接近0)                      | 否 (输出恒非负)                       | 否 (但比ReLU更接近)                 | 否 (但比ReLU更接近)                | 是 (输出均值更接近0)                 | 否 (输出恒正)                           |
| **导数**         | `σ'(x) = σ(x)(1-σ(x))`                  | `tanh'(x) = 1 - tanh^2(x)`              | `1 if x>0, 0 if x<0`                  | `1 if x>0, α if x<0`                | `1 if x>0, α if x<0`               | `1 if x>0, f(x)+α if x≤0`            | 复杂 (雅可比矩阵)                     |
| **计算效率**     | 中 (含指数运算)                         | 中 (含指数运算)                         | 非常高 (只需比较)                     | 非常高 (比较和乘法)                   | 非常高 (比较和乘法)                  | 高 (负数部分含指数运算)              | 中 (含指数和除法运算)                 |
| **主要优点**     | 1. 输出在(0,1)间, 可解释为概率<br>2. 平滑，处处可导 | 1. 输出零中心<br>2. 比Sigmoid收敛更快 | 1. 计算高效<br>2. 缓解梯度消失(正区间)<br>3. 促进稀疏性 | 1. 解决ReLU的“神经元死亡”问题<br>2. 计算高效 | 1. 解决“神经元死亡”<br>2. 负斜率可自适应学习 | 1. 解决“神经元死亡”<br>2. 输出均值近零<br>3. 可能有更好的性能 | 1. 输出可解释为多分类概率<br>2. 适用于多分类任务 |
| **主要缺点**     | 1. 梯度消失问题<br>2. 输出非零中心<br>3. 计算相对复杂 | 1. 梯度消失问题 (比Sigmoid稍好)<br>2. 计算相对复杂 | 1. “神经元死亡”问题 (Dying ReLU)<br>2. 输出非零中心 | 1. α选择需经验/实验<br>2. 性能提升不保证 | 1. 增加了模型参数<br>2. 可能过拟合α | 1. 计算比ReLU复杂 (负数部分)           | 1. 仅用于输出层<br>2. 计算相对复杂       |
| **常见应用**     | 1. 二分类输出层<br>2. RNN/LSTM门控 (历史) | 1. 隐藏层 (比Sigmoid更优)<br>2. RNN/LSTM门控 | 1. CNN、MLP等网络的隐藏层 (常用默认) | 1. ReLU的替代方案                      | 1. ReLU/Leaky ReLU的替代方案        | 1. ReLU的替代方案, 有时性能更好       | 1. 多分类问题的输出层                 |
| **梯度消失**     | 严重                                    | 存在                                    | 正区间不存在, 负区间梯度为0             | 基本解决                              | 基本解决                             | 基本解决                             | 可能存在 (输入差异大时)             |


**如何选择激活函数？**
- **没有绝对“最好”的激活函数**：选择通常取决于模型架构、具体任务和实验效果。
- **隐藏层（内部特征提取）**：
  - **传统网络与视觉模型（MLP / CNN）**：通常建议从 **ReLU** 开始（计算最快、收敛迅速）；若怀疑出现“神经元死亡”（部分神经元梯度恒为 0 不再更新），可尝试 **Leaky ReLU / PReLU / ELU**。
  - **Transformer 与大语言模型（LLM）**：主流标配已转向更平滑的 **GELU**（如 BERT、GPT）或带门控机制的 **SwiGLU / SiLU**（如 DeepSeek、Qwen、LLaMA），有助于深层网络的稳定训练。
- **输出层（由任务目标决定）**：
  - **二分类 / 多标签分类**：用 **Sigmoid**（将输出压缩为独立的 $0 \sim 1$ 概率）。
  - **互斥多分类**：用 **Softmax**（将输出转化为总和为 $1$ 的概率分布）。
  - **回归任务（预测连续值）**：**不加激活函数**（直接输出线性值 Linear）。


实际上的神经网络为了处理更多类型的复杂任务，人们设计了更为复杂的架构，这些架构在不同任务的表现上，发挥了不同的作用。

早期最常见的神经网络层类型有：
- **Fully connected layer (全连接层)**：在全连接层中，每个神经元都连接到前一层和下一层中的每个神经元，从而允许输入和输出之间的复杂关系。    
- **Convolutional layer (卷积层)**：卷积层使用过滤器从输入数据中提取空间特征，从而在图像或顺序数据中实现有效的模式识别。
- **Recurrent layer (递归层/循环层)**：循环神经网络层通过内置记忆单元来处理序列数据，能够捕捉先前输入之间的依赖关系和交互作用，因此非常适合时间序列分析或自然语言处理等任务。

#### 2.1.2 训练核心目标：找到最佳“旋钮组合”

神经网络是由相互连接的“神经元”（计算单元）组成的网络。这些连接有不同的强度，称为权重（Weights），每个神经元还有一个偏置（Bias）。训练的目标就是调整网络中所有的权重和偏置，使得对于给定的输入（例如图片），网络能产生尽可能接近真实目标（例如“猫”的标签）的输出。就像调试一台复杂的收音机，我们需要旋转很多旋钮（调整权重和偏置），直到能清晰地收到目标电台（得到正确的预测）。

简单进行形式化表示就是，**给定一个数据集，包含$N$个输入输出的例子（称为样本）：$D=\{(x_i,y_i)\}_{i=1}^N$，我们希望找到一组参数$\omega$，使模型预测的输出$\hat{y}=F(x;\omega)$与真实的输出$y$，尽可能的接近。**

#### 2.1.3 如何衡量“好坏”？—— 损失函数 (Loss Function)

在我们调整“旋钮”之前，需要一个标准来衡量当前网络的表现有多“差”。这就是损失函数的作用。
- 定义： 它计算网络预测值（Prediction）与真实目标值（True Target）之间的差异。差异越大，损失值越高。
- 目标： 训练过程的目标就是最小化损失函数的值。

常见损失函数：
- 均方误差 (Mean Squared Error, MSE): 常用于回归问题（预测连续值，如房价）。
   - 公式：$Loss = (1/n) * Σ(y_{pred} - y_{true})^2$ (n是样本数量, $y_{pred}$是预测值, $y_{true}$是真实值)
- 交叉熵损失 (Cross-Entropy Loss): 常用于分类问题（预测离散类别，如猫/狗）。

#### 2.1.4 朝哪个方向调整？—— 梯度 (Gradient)

知道了当前的表现有多差（损失），下一步是如何调整权重和偏置来改进它？首先我们需要知道调整的方向（“方向比努力更重要”😊）。
- 定义： 梯度是损失函数相对于每个权重和偏置的偏导数。它表示了如果我们稍微改变某个参数，损失函数会如何变化。对于多变量函数，梯度是一个向量，包含了函数在该点处沿着各个变量方向的偏导数。

$$\nabla_{\omega}L(D,\omega)=\left[\frac{\partial L(D,\omega)}{\partial \omega_1},\frac{\partial L(D,\omega)}{\partial \omega_2},\cdots,\frac{\partial L(D,\omega)}{\partial \omega_M}\right]$$

![partial_derivative](https://drive.google.com/uc?id=1cFzmHDs0SYdqPQcOk-yhnp72KldKlX2D)

- 关键点： 梯度指向损失函数增长最快的方向。
- 类比： 你站在一座山上（损失函数曲面），梯度告诉你哪个方向是上坡最陡峭的路。

#### 2.1.5 如何一步步改进？—— 梯度下降 (Gradient Descent)

既然梯度指向损失增加最快的方向，那我们只需要朝着梯度相反的方向调整参数，就能让损失减小！这就是梯度下降的核心思想。

<img src="https://drive.google.com/uc?id=19OkD8ZOdn5GDPofG81qs9X0LAQzlTv33" width="400px">

过程：
1. 选择一个初始的权重和偏置（通常是随机的）。
2. 计算当前参数下损失函数的梯度。
3. 沿着梯度的负方向更新参数。
4. 重复步骤 2 和 3，直到损失不再显著下降（达到最小值或一个可接受的低点）。

更新规则：
- $new_{weight} = old_{weight} - learning\_rate * gradient_{loss\_wrt\_weight}$
- $new_{bias} = old_{bias} - learning\_rate * gradient_{loss\_wrt\_bias}$

#### 2.1.6 步子迈多大？—— 学习率 (Learning Rate, $\alpha$)

在梯度下降的更新规则中，有一个重要的超参数（需要手动设置的参数）：学习率。
- 定义： 它控制了每次更新参数时“步子”的大小。
- 影响：
  - 学习率过小： 训练速度会非常慢，可能需要很多很多步才能收敛。
  - 学习率过大： M可能会在最小值附近“震荡”甚至“飞出”最优区域，导致无法收敛或损失变大。

![lr](https://drive.google.com/uc?id=12iOjT44NNtW_FXBlkEQKGNVdptVNuhqv)

学习率的选择非常关键，通常需要实验来找到一个合适的值（例如 0.01, 0.001, 0.0001）。

#### 2.1.7 如何高效计算所有梯度？—— 反向传播 (Backpropagation)

对于一个有很多层、很多参数的神经网络，手动计算每个参数的梯度是非常复杂且低效的。反向传播是一种强大的算法，它能高效地计算损失函数相对于网络中所有参数（权重和偏置）的梯度。反向传播是现代深度学习训练的基石，它使得训练深度、复杂的网络成为可能。

核心思想（基于链式法则）：
- 前向传播 (Forward Pass): 输入数据通过网络，计算每一层的输出，直到最终得到预测值，然后计算损失。
- 反向传播 (Backward Pass): 从输出层开始，计算损失对该层输出的梯度。然后，利用链式法则，将这个梯度反向传播到前一层，计算损失对前一层输出和连接权重的梯度。一层一层地向后传递，直到计算出损失对所有参数的梯度。

<img src="https://drive.google.com/uc?id=19b394EUtyU4j7Qpr-cnsFKmkSJHurTZw" width="600px">

> **🤔 为什么反向传播很高效？**
> * 逆向设计：因为大模型的终点是一个单一的 Loss。从一到多（反向倒推），永远比从多到一（正向挨个测试）高效得多。
> * 结果复用：运用链式法则，通过一次前向传播计算并存储中间激活值，在回溯过程中，把上一层算好的梯度存成了缓存，下一层直接调用，消除了所有的重复计算。

#### 2.1.8 更聪明的下降方法—— 求解器/优化器 (Optimizer)

基础的梯度下降（有时称为“批量梯度下降”，Batch GD）虽然有效，但在大型数据集上可能很慢，并且容易陷入局部最小值（看起来是最低点，但不是全局最低点）。

<table><tr>
<td><img src="https://drive.google.com/uc?id=1hRV-gR_yP4pckwZMyzPgFP7OMpUEdxkJ" width="400px"></td>
<td><img src="https://drive.google.com/uc?id=13uc7ppSZ3n2CH1PDZCYkZ9q_LjtCkjzA" width="400px"></td>
</tr></table>

优化器是改进梯度下降过程的算法，旨在更快、更稳定地找到最小值。

常见优化器：
- 随机梯度下降 (Stochastic Gradient Descent, SGD): 每次只用一个或一小批（mini-batch）数据计算梯度并更新，速度快，引入的随机性有助于跳出局部最小值。
- Momentum: 在更新时加入“动量”项（基于之前的更新方向），有助于加速收敛并越过小的沟壑。
- Adam (Adaptive Moment Estimation): 结合了 Momentum 和 RMSprop (另一种自适应学习率方法) 的优点，为每个参数计算自适应的学习率。通常是默认首选的优化器，效果稳健。

<img src="https://drive.google.com/uc?id=1z8Nr4UonZmCBhmVd0YIVu_oXHiFec6qU" width="900px">

可视化例子（来源：[Stanford CS231n](https://cs231n.github.io/neural-networks-3/)）

<table><tr>
<td><img src="https://drive.google.com/uc?id=1wfNfyqPgw0bViD7mJgugJFqBhTJPIazY" border=0></td>
<td><img src="https://drive.google.com/uc?id=1xJmcn3W5h2BYe7hXKZnrhR0ceQpX5BZR" border=0></td>
</tr></table>

#### 2.1.9 训练中的挑战—— 梯度消失 (Vanishing Gradients)

在很深的网络中（有很多层），使用某些激活函数（如 Sigmoid 或 Tanh）时，反向传播计算出的梯度在逐层传递过程中可能会越来越小，接近于零。使得靠近输入层的权重几乎接收不到梯度信号，导致它们更新非常缓慢或停止更新，网络难以学习。

原因是，Sigmoid 和 Tanh 函数在其饱和区域（输入值很大或很小时）的导数非常接近于零。反向传播中链式法则涉及多个这种小导数的连乘，结果会迅速趋近于零。

假设我们有一个10层的神经网络，每层使用 sigmoid 激活函数：
- sigmoid 函数的导数最大值是 0.25
- 在反向传播时，这些导数会相乘
- 0.25^10 ≈ 0.0000095
- 这意味着即使最后一层的梯度是1，传到第一层时也只剩下约0.0000095

<img src="https://drive.google.com/uc?id=1gccBmMLNhjIqvmiZv1xG_djCOLn3y4Jl" style="margin-left: 0px" width="700px">

图源：斯坦福CS224n

解决方案：
- 使用 ReLU (Rectified Linear Unit) 及其变种 (Leaky ReLU, ELU) 作为激活函数，它们的导数在正区间是常数 1，缓解了梯度消失。
- 使用 Batch Normalization。
- 使用残差连接 (Residual Connections)，如 ResNet 架构。也可以采用门控机制（LSTM、GRU）。
- 更优的权重初始化方法。

#### 2.1.10 其他模型训练常见问题及应对策略(扩展学习)

| 问题 | 问题说明 | 发生原因 | 引起的后果 | 解决策略 |
|------|----------|-----|-------|----------|
| **梯度消失** | • 梯度在反向传播过程中逐层减小<br>• 深层网络中梯度接近于0<br>• 常见于使用 sigmoid 等饱和激活函数时 | • sigmoid/tanh 等激活函数在饱和区梯度接近0<br>• 多层网络中梯度连乘导致数值越来越小<br>• 权重初始化不当导致激活值落入饱和区<br>• 网络层数过深造成梯度信息衰减 | • 深层参数几乎不更新<br>• 模型训练停滞<br>• 深层特征无法学习 | • 使用 ReLU 等非饱和激活函数<br>• 添加残差连接（Skip Connection）<br>• 使用批量归一化（Batch Normalization）<br>• 合适的权重初始化<br>• LSTM/GRU 等门控机制 |
| **梯度爆炸** | • 梯度值异常增大<br>• 权重更新幅度过大<br>• 常见于深层网络或 RNN | • 权重矩阵特征值大于1导致梯度指数增长<br>• 学习率设置过大<br>• 数据未经归一化处理<br>• 网络结构不合理导致梯度累积 | • 模型参数变得极大<br>• 训练不稳定<br>• 可能导致 NaN 错误 | • 梯度裁剪（Gradient Clipping）<br>• 权重正则化<br>• 使用 ReLU 激活函数<br>• 合适的学习率<br>• 批量归一化 |
| **过拟合** | • 模型过度拟合训练数据<br>• 学习到数据中的噪声<br>• 模型复杂度过高 | • 训练数据量不足<br>• 模型参数量相对于数据量过大<br>• 训练时间过长<br>• 数据质量问题（噪声、异常值）<br>• 缺乏有效的正则化措施 | • 测试集性能差<br>• 泛化能力弱<br>• 对新数据预测不准 | • 数据增强<br>• Dropout 正则化<br>• L1/L2 正则化<br>• 提前停止（Early Stopping）<br>• 减小模型复杂度 |
| **欠拟合** | • 模型复杂度不够<br>• 特征提取能力不足<br>• 训练不充分 | • 模型结构过于简单<br>• 特征工程不足<br>• 正则化强度过大<br>• 训练轮数不够<br>• 学习率设置不当 | • 训练集和测试集都表现差<br>• 无法捕捉数据特征<br>• 预测效果差 | • 增加模型复杂度<br>• 增加训练轮数<br>• 调整学习率<br>• 添加特征工程<br>• 减少正则化强度 |
| **不平衡数据** | • 各类别样本数量差异大<br>• 少数类样本不足<br>• 模型偏向多数类 | • 数据收集过程中的固有偏差<br>• 现实世界中类别本身就不平衡<br>• 标注成本限制<br>• 数据获取难度不同 | • 少数类预测效果差<br>• 评估指标不准确<br>• 模型实用性差 | • 过采样（SMOTE 等）<br>• 欠采样<br>• 类别权重调整<br>• 生成对抗网络（GAN）<br>• 集成学习 |
| **训练不稳定** | • 损失函数波动大<br>• 优化过程不收敛<br>• 性能不一致 | • 学习率设置不当<br>• 批量大小选择不合适<br>• 数据预处理不充分<br>• 损失函数设计不合理<br>• 优化器选择或配置不当 | • 训练难以进行<br>• 结果不可复现<br>• 模型性能不稳定 | • 批量归一化<br>• 学习率调度<br>• 梯度累积<br>• 使用优化器（Adam）<br>• 权重初始化 |
| **内存问题** | • GPU/内存资源不足<br>• 批量大小受限<br>• 模型规模受限 | • 模型参数量过大<br>• 中间激活值占用内存过多<br>• 批量大小设置过大<br>• 数据预处理占用内存<br>• 多GPU训练通信开销 | • 无法训练大模型<br>• 训练速度慢<br>• 批量大小受限 | • 梯度检查点<br>• 混合精度训练<br>• 模型并行<br>• 梯度累积<br>• 模型压缩 |
| **超参数调优** | • 参数选择困难<br>• 调优过程耗时<br>• 参数组合众多 | • 超参数之间存在复杂交互<br>• 参数敏感度不同<br>• 计算资源限制<br>• 缺乏理论指导<br>• 任务特异性强 | • 模型性能不佳<br>• 训练效率低<br>• 资源浪费 | • 网格搜索<br>• 随机搜索<br>• 贝叶斯优化<br>• 自动化调参<br>• 交叉验证 |

注意事项：
1. 问题之间可能相互关联，一个问题的解决方案可能对其他问题也有帮助
2. 解决策略的选择需要考虑具体场景和资源限制
3. 建议采用渐进式方法，从简单的解决方案开始尝试
4. 保持对训练过程的监控，及时发现和解决问题

<div class="alert alert-success">
<b>训练流程概览：</b>
    <ul>
        <li>初始化： 随机初始化网络的权重和偏置。</li>
        <li>迭代循环 (Epochs): 对整个训练数据集重复以下步骤多次：</li>
            <ul>
                <li>选择批次 (Batch): 从训练数据中取一小批样本。</li>
                <li>前向传播：将批次数据输入网络，计算得到预测值。</li>
                <li>计算损失：使用损失函数计算预测值与真实值之间的误差。</li>
                <li>反向传播：计算损失函数相对于网络中所有参数的梯度。</li>
                <li>参数更新：使用优化器（如 Adam），根据计算出的梯度和学习率，更新网络的权重和偏置（参数 = 参数 - 学习率 * 梯度 的变种）。</li>
            </ul>
        <li>评估： 在训练过程中或结束后，使用独立的验证集或测试集评估模型性能。</li>
        <li>停止： 当模型性能不再提升（或达到预设的迭代次数）时，停止训练。</li>
    </ul>
</div>

> **ANN 的问题:**
> 1. <strong>无法处理序列数据:</strong> 标准的 MLP 假设输入是固定大小的向量，并且输入单元之间没有时序关系。它无法捕捉文本、语音等序列数据中的顺序依赖性（例如，一个词的含义受前面词的影响）。
> 2. <strong>无记忆能力:</strong> 对于每个输入，模型的处理是独立的，没有“记忆”之前处理过的信息。


### 2.2 循环神经网络 (Recurrent Neural Network, RNN)

为了处理序列数据而设计的神经网络。RNN 引入了 **循环连接**，使得当前时间步的输出不仅依赖于当前输入，还依赖于 **前一时间步的隐藏状态 (Hidden State)。** 这个隐藏状态就像模型的“记忆”，能够编码之前看到的信息。RNN 的核心思想是“记忆”：它会把前面时间步处理过的信息，传递给后面的时间步。就像我们读书，会记住前面句子的内容，才能理解后面的意思。

![rnn](https://drive.google.com/uc?id=1UQquIyF4FkxTnpHxYnkaPYq3vBMoueqt)
<center>RNN 展开图 (图片来源: Wikimedia Commons)</center>

图中 $X_t$ 是当前时间步的输入，$h_t$ 是当前时间步的隐藏状态（可以理解为“记忆”），$O_t$ 是输出。关键在于 $h_t$ 不仅取决于 $X_t$，还取决于上一个时间步的隐藏状态 $h_{t-1}$。

> 请大家仔细看图里的三个英文字母：**$U$、$W$、$V$**。不管是在 $t-1$、还是 $t$、还是 $t+1$，**这三个字母上，都没有写任何下标。为什么？**
>
> 这代表了循环神经网络一个很重要的数学设定：**参数共享（Weight Sharing）**。
> * 这意味着不管这个句子有多长，大模型用来理解输入（$U$）、传递记忆（$V$）、做出预测（$W$）的，是完全**同一套参数！**
> * 这极大地节省了大模型的参数量。这就好比你读一万本书，你脑子里“理解语言的语法规则（参数）”是不需要变的，变动的只是随着时间流逝、输入进来的具体剧情（数据）而已。

看起来似乎还行，对吧？但 RNN 在实践中遇到了一些棘手的问题。我们来逐一看看。

**问题一：梯度消失 (Vanishing Gradients) 与长期依赖困难**

这是 RNN 最著名也最麻烦的问题。梯度是模型学习时用来更新参数的“指示信号”，在 RNN 处理长序列时，这个“指示信号”往回传的过程中，可能会变得越来越弱，就像悄悄话游戏，传到最后几乎听不清了。当信号弱到接近 0 时，靠前的那些时间步的计算就几乎收不到有效的更新指示了。
因为在反向传播过程中，梯度会经过很多次矩阵乘法和激活函数的导数。如果这些乘数在很多步上都小于 1，那么梯度就会以指数级速度衰减。

梯度消失导致 RNN **难以学习“长期依赖”关系**。比如这句话：
> "我在法国长大，并在那里度过了我的童年...... 因此，我能说流利的 <font color="red">**法语**</font>。"

模型需要记住很久之前的 "法国"，才能准确预测出最后的 "法语"。如果梯度消失了，关于 "法国" 的信息（对应的梯度信号）传不到后面，模型就很难学会这种联系。RNN 的“记忆”变得很短暂。

**问题二：梯度爆炸 (Exploding Gradients)**

这个是梯度消失的反面，但也同样麻烦。指示信号在反向传播时，不但没有减弱，反而指数级地增大了，变得异常巨大。梯度爆炸跟梯度消失类似，也是因为链式法则的连乘效应。但这次是因为权重 Wh 的某些部分或激活函数导数在很多步上都**大于 1**。

梯度爆炸会导致模型训练**极其不稳定**。巨大的梯度会让参数更新步子迈得太大，可能直接“跨过”最优解，甚至让损失函数变成 NaN（Not a Number，表示无效数字）。梯度爆炸相对容易处理一些。常用一种叫做**梯度裁剪 (Gradient Clipping)** 的技术：设定一个梯度的上限，如果计算出来的梯度超过这个上限，就强行把它“拽”回上限值。这能防止更新步子过大，保证训练的基本稳定，但它更像是“治标不治本”。

为了缓解 RNN 的梯度消失/爆炸问题，发展出了更复杂的门控 RNN 单元，如 **长短期记忆网络 (LSTM, Hochreiter & Schmidhuber, 1997)** 和 **门控循环单元 (GRU, Cho et al., 2014)，** 它们通过引入“门”机制来更好地控制信息的流动和遗忘，从而捕捉更长距离的依赖。

1.  **序列建模:** 通过循环连接和隐藏状态，能够处理变长的序列输入，并捕捉局部的时间依赖关系。
2.  **短期记忆:** 隐藏状态提供了对过去信息的有限记忆。

<img src="https://drive.google.com/uc?id=1boZRUkKDADXDuP1AJBnKOdNus8im7Jym">


**问题三：计算效率低下 (Sequential Computation)**

这个问题根植于 RNN 的核心工作方式。**RNN 的计算是“串行”的：**

为了计算时间步 `t` 的隐藏状态 $h_t$，你**必须**先得到时间步 `t-1` 的隐藏状态 $h_{t-1}$。这意味着，你无法同时计算所有时间步的状态。必须像多米诺骨牌一样，一个接一个地算：先算第 1 步，得到 $h_1$；再用 $h_1$ 算第 2 步，得到 $h_2$；以此类推... 直到序列结束。想象一下，建一座高塔，你必须一层一层地往上盖，不能所有层同时开工。RNN 处理序列也是类似。

这就导致：
- **训练和推理速度慢：** 特别是对于很长的序列，计算时间会随着序列长度线性增加。
- **难以并行化：** 现代计算设备（GPU/TPU）的强大之处在于并行计算。但 RNN 的这种串行依赖性，使得在**时间维度**上很难进行大规模并行处理，限制了硬件效率的发挥。

#### Seq2Seq 模型 (以下内容，主要来自于Jay Alammar的[一篇博客文章](https://jalammar.github.io/visualizing-neural-machine-translation-mechanics-of-seq2seq-models-with-attention/))

Seq2Seq 就是从一个序列到另一个序列的意思，典型的任务包括：机器翻译，文本摘要。Google 在 2016 年末开始使用 Seq2Seq 模型进行机器翻译，取得了很好的效果，并发表了 2 篇开创新论文：
- Sutskever等，2014年发表的 [Sequence to Sequence Learning with Neural Networks](https://papers.nips.cc/paper/5346-sequence-to-sequence-learning-with-neural-networks.pdf)
- Cho等，2014年发表的 [Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation](http://emnlp2014.org/papers/pdf/EMNLP2014179.pdf)

![seq2seq](https://drive.google.com/uc?id=19h35Bt9Y8QnXfcc8eS9JIGYYepuh018s)

上图展示了 Seq2Seq 模型的机器翻译的例子。Seq2Seq 模型中的编码器和解码器一般采用的是循环神经网络 RNN（Transformer 模型还没出现）。在 Seq2Seq 模型中使用 encoder 进行编码，使用 decoder 进行解码。编码器会将输入的一个句子的词元序列编码成一个 context 向量，然后传递给解码器，解码器根据 context 向量解码出翻译后的句子序列。

编码器的执行过程如下：

1. 假设序列输入是一个句子，这个句子可以由$n$个词表示：$sentence = {w_1, w_2,...,w_n}$。
2. RNN 首先将句子中的每一个词映射成为一个向量得到一个向量序列：$X = {x_1, x_2,...,x_n}$。
3. 然后在处理第 $t \in [1,n]$ 个时间步的序列输入 $x_t$ 时，RNN 的输入和输出可以表示为：$h_{t} = RNN(x_t, h_{t-1})$
   - 输入：RNN在时间步 $t$ 的输入之一为单词 $w_t$ 经过映射得到的向量 $x_t$。
   - 输入：RNN 另一个输入为上一个时间步 $t-1$ 得到的 hidden state 向量 $h_{t-1}$，同样是一个向量。
   - 输出：RNN 在时间步 $t$ 的输出为 $h_t$ hidden state向量。

最后一步 $h_n$ 就是整个句子的 context 向量。

![seq2seq RNN](https://drive.google.com/uc?id=1bOtIOY2HMmBNilbEKMV73pRHc6eZjPRH)

解码器的执行步骤也是类似，每执行一步都会将隐藏层状态传递给下一步。

问题在于，基于 RNN 的这个向量计算的递归过程，本质上是对这个词元构成的序列中的信息进行了向量的压缩，最终的 context 维度是不变的，这样就必然会导致信息损失。而且，递归的过程很长的情况下（长文本处理），信息损失会加剧，导致模型性能下降（遗忘）。面对这些问题，Bahdanau 等 2014 发布的 [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473) 和 Luong等 2015年发布的 [Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/abs/1508.04025) 两篇论文中，提出了一种叫做注意力 `attetion` 的技术。通过 `attention` 技术，Seq2Seq 模型极大地提高了机器翻译的质量。归其原因是：**Attention 注意力机制，使得 Seq2Seq 模型可以有区分度、有重点地关注输入序列**。

> **怎么理解 RNN 和 Seq2Seq 的区别？**
>
> 打个比方：RNN 像一个人边听故事边记笔记，每次听到新句子，就更新自己的记忆（隐藏状态），但记性差，听到后面容易忘掉开头细节（长程依赖问题）；Seq2Seq 像两个人合作，编码器（听故事）先把整个故事消化成「精华要点」（上下文向量），解码器（讲故事）拿着这个要点，用自己的话复述出来。加了注意力机制后，复述时会反复翻看原始重点（像考试划重点）。

### 2.3 注意力机制 (Attention Mechanism)

注意力机制的核心思想是模仿人类在处理信息时的选择性注意。比如当我们阅读一个句子时，会根据上下文对不同的词赋予不同的关注度。

<div class="alert alert-block">
<font color="red" size="6">研表究明，汉字的序顺并不定一会影阅响读。比如当你看完这句话后，才发这现里的字全是都乱的。</font>
</div>

眼动实验证明，我们在阅读文字的过程中**并不是看完整的句子，而是看关键字**。因为对文字的熟悉和记忆，**大脑知道哪些文字代表关键信息**，当照在这些关键字上的光反射进入眼球时，大脑迅速作出识别并丢弃了**不重要的信息**。

大语言模型对这句话的解读：

![Attention](https://drive.google.com/uc?id=1KjXjXN96EIZmKoLJCye_RhHJ0z7HNWlj)


Attention 最早被引入神经机器翻译 (NMT) 领域 (Bahdanau et al., 2014)，用于解决基于 RNN 的 Seq2Seq 模型（编码器-解码器结构）的**信息瓶颈**问题。

传统的 Seq2Seq 模型将编码器对整个输入序列的理解压缩成一个**固定大小的上下文向量 (Context Vector)**，然后解码器仅基于这个向量生成输出序列。对于长输入序列，这个固定大小的向量很难承载所有重要信息，导致信息丢失。
- **Attention 机制允许解码器在生成每个输出词时，能够“关注”输入序列的不同部分，并根据相关性赋予不同的权重**。它会计算一个“注意力分数”，表示当前解码步骤应该多关注输入序列的哪些词。然后，通过对编码器各时间步隐藏状态的加权求和，动态地生成一个**针对当前解码步的上下文向量**。
>  Attention 机制的核心思想是：**不要一字一句地读完句子再思考，而是直接去看与当前任务最相关的单词。**

主要改进在两处：
1. 在编码器中，不再只使用最后一个隐藏状态作为 context 向量，而是把所有时间步的 hidden state（隐藏层状态）传递给解码器。
2. 注意力模型的解码器在产生输出之前，做了一个额外的attention处理：
   - 给每个 hidden state $h_t$ 计算出一个分数（这个分数可以通过点积、加权和或拼接等方法计算）。
   - 所有hidden state（隐藏层状态）的分数经过 softmax 进行归一化。
   - 将每个 hidden state（隐藏层状态）乘以所对应的分数，从而能够让高分对应的 hidden state（隐藏层状态）会被放大，而低分对应的 hidden state（隐藏层状态）会被缩小。
   - 将所有hidden state根据对应分数进行加权求和，得到对应时间步的 context 向量。
   
最终带 Attention 的解码器的过程如下图所示。

![seq2seq Attention](https://drive.google.com/uc?id=1npv8ZXu4PjXWR_iRcabiBoWylR6dzGvC)

1. 注意力模型的解码器 RNN 的输入包括：一个 embedding 向量，和一个初始化好的解码器 hidden state，图中是 $h_{init}$。
2. RNN 处理上述的 2 个输入，产生一个输出和一个新的 hidden state，图中为$h_4$。
3. 注意力的步骤：我们使用编码器的所有 hidden state向量和 $h_4$ 向量来计算这个时间步的 context 向量（$C_4$）。
4. 我们把 $h_4$ 和 $C_4$ 拼接起来，得到一个橙色向量。
5. 我们把这个橙色向量输入一个前馈神经网络（这个网络是和整个模型一起训练的）。
6. 根据前馈神经网络的输出向量得到输出单词：假设输出序列可能的单词有N个，那么这个前馈神经网络的输出向量通常是 N 维的，每个维度的下标对应一个输出单词，每个维度的数值对应的是该单词的输出概率。
7. 在下一个时间步重复1-6步骤。

**优势：**
1.  **动态上下文:** 为解码器的每一步生成不同的、更相关的上下文向量。
2.  **缓解长距离依赖问题:** 解码器可以直接访问并关注输入序列的任何部分，不受距离限制。
3.  **可解释性:** 注意力权重可以一定程度上可视化，了解模型在生成某个输出时关注了输入的哪些部分。

### 2.4 Transformer (Vaswani et al., 2017, "Attention Is All You Need")

Transformer 是一个里程碑式的架构（如下图），**完全摒弃了 RNN 的循环结构和 CNN 的卷积操作，仅依赖于注意力机制（特别是自注意力 Self-Attention）来捕捉序列内的依赖关系**。

![Transformer](https://drive.google.com/uc?id=16M6-iBg0i7bg2ncQZ4eUzYPdHpcGEFsw)

Transformer 沿用了经典的 Seq2Seq（序列到序列）模型的 **Encoder-Decoder** 框架。模型结构中的左半部分为编码器（encoder），右半部分为解码器（decoder）。

**Encoder (编码器):**
- 负责理解输入序列（如源语言句子）。
- 由 **N** 个相同的 **Encoder Layer** 堆叠而成。
- 将输入序列转换为一系列上下文向量。

**Decoder (解码器):**
- 负责生成输出序列（如目标语言句子）。
- 由 **N** 个相同的 **Decoder Layer** 堆叠而成。
- 在生成每一个输出词时，会参考 Encoder 的输出以及已经生成的部分序列。

> 🤔 **Shifted right outputs 是什么意思？为什么要这么设计？**
>
> "Outputs (shifted right)" 是 Transformer 解码器（Decoder） 在**训练阶段**使用的一种特殊输入。它的目的是让解码器学习在给定前面已生成的正确词元（token）的情况下，预测下一个正确的词元。
> * **模拟生成过程（Teacher Forcing）**: 解码器的任务是自回归（auto-regressive）地生成输出序列，即预测下一个词元需要依赖之前已经生成的所有词元。在训练时，我们希望模型在每一步都能得到正确的“前文”信息来学习预测“下一个”词元。使用 "Outputs (shifted right)" 作为输入，就是强制给模型提供 正确 的前文信息（这称为 Teacher Forcing），让它专注于学习从 (< bos >, y1, ..., yi-1) 到 yi 的映射关系。
> * **并行计算**: 虽然生成过程是串行的，但在训练时，我们可以利用目标序列已知这一优势。通过将右移后的整个目标序列一次性输入解码器，并配合 Masked Self-Attention（掩码自注意力） 机制（确保在预测位置 i 时只能关注 i 之前的位置），模型可以并行地计算所有位置的预测损失，极大地提高了训练效率。


**核心组件:**
   - **位置编码 (Positional Encoding):** 由于没有循环结构，Transformer 本身无法感知词的顺序。因此，需要向输入嵌入中加入位置编码信息，让模型知道词在序列中的位置。常用的位置编码方法有绝对位置编码和旋转位置编码（RoPE, Su, Jianlin, et al. 2024）。
   - **自注意力 (Self-Attention):** 允许模型在处理序列中的一个词时，同时计算该词与序列中所有其他词（包括自身）的关联程度（注意力权重），然后加权聚合所有词的信息来生成该词的新表示。这使得模型能直接捕捉长距离依赖，且计算可以高度并行化。
   - **多头注意力 (Multi-Head Attention):** 将 Self-Attention 并行地运行多次（多个“头”），每个头关注不同的表示子空间信息，然后将结果拼接起来。多头注意力机制可以理解为，不是用一套参数（一个“头”）去计算词语之间的相关性，而是同时使用多套独立的参数（多个“头”）并行地进行注意力计算，这就像让模型从不同的角度或不同的‘视角’去看待输入序列中的词语，每个‘头’都能学会捕捉一种不同类型的关联信息（如语法关系、语义关系），通过将这些不同‘视角’下得到的注意力结果结合起来，模型能够更全面、更深入地理解序列中词语之间的复杂关系，从而获得更丰富、更有效的上下文表示。
   
**优势：**
1.  **并行计算:** Self-Attention 的计算不依赖于前一时间步的结果，可以在序列长度维度上完全并行，大大加快了训练速度，使得训练更大模型成为可能。
2.  **有效捕捉长距离依赖:** Self-Attention 直接计算任意两个位置之间的关联，信息传递路径最大长度为 O(1)，远优于 RNN 的 O(N)。
3.  **成为后续大型模型的基础:** Transformer 架构的强大能力和可扩展性，使其成为 BERT、GPT 等现代 LLM 的基石。

#### 2.4.1. Encoders & Decoders

![Encoders & Decoders](https://drive.google.com/uc?id=1xuzWeBWq7p9cgxjQir6AhDx3LQwBJtOI)

![Encoders & Decoders](https://drive.google.com/uc?id=16kVlbc9Y0Hcfk54U5B1YBGweIgK6C_P0)

**单层 Encoder**

单层 Encoder 由两个子层组成：
- Self-Attention Layer
- Feed-Forward Neural Network (FFNN)

编码器的输入文本序列 $w_1, w_2,...,w_n$ 最开始需要经过 embedding 转换，得到每个单词的向量表示 $x_1, x_2,...,x_n$，其中$x_i \in \mathbb{R}^{d}$ 是维度为 $d$ 的向量，然后所有向量经过一个 Self-Attention 神经网络层进行变换和信息交互得到 $h_1, h_2,...h_n$，其中 $h_i \in \mathbb{R}^{d}$ 是维度为 $d$ 的向量。self-attention 层处理一个词向量的时候，不仅会使用这个词本身的信息，也会使用句子中其他词的信息（你可以类比为：当我们翻译一个词的时候，不仅会只关注当前的词，也会关注这个词的上下文的其他词的信息）。Self-Attention 层的输出会经过前馈神经网络得到新的 $x_1, x_2,..,x_n$，依旧是 $n$ 个维度为 $d$ 的向量。这些向量将被送入下一层 encoder，继续相同的操作。

**单层 Decoder**

单层 Decoder 由三个子层组成：
- Self-Attention Layer
- Encoder-Decoder Attention Layer：这一层的作用是让 Decoder 的每个位置都能关注到输入序列的所有位置。
- Feed-Forward Neural Network (FFNN)

![single layer](https://drive.google.com/uc?id=1JQ-lamO8UocOxe4M2_I3-m16ThE0HcJQ)

在 Transformer 论文中，展示的是 6 层编码器，这不是固定不变的。GPT 3.5 使用了 96 层的 Decoder-only Transformer 模型结构。

编码器一般有多层，第一个编码器的输入是一个序列文本，最后一个编码器输出是一组序列向量，这组序列向量会作为解码器的K、V输入，这些注意力向量将会输入到每个解码器的Encoder-Decoder Attention层。

解码（decoding）阶段的每一个时间步都输出一个翻译后的单词（这里的例子是英语翻译），解码器当前时间步的输出又重新作为输入Q和编码器的输出K、V共同作为下一个时间步解码器的输入。然后重复这个过程，直到输出一个结束符。如下图所示：

![Decoder](https://drive.google.com/uc?id=1_8MMylW7JkgF1Re3Ad6HH38A9PxJqdim)

Transformer 在输入的时候，作了一个处理：在 Input Embedding 上，加了一个 Positional Encoding。这个 Positional Encoding 是为了让 Transformer 模型能够区分不同位置的词，因为 Transformer 模型是没有循环的，所以没有办法通过时间步来区分不同位置的词。

> 🤔 **请回忆一下上面的 RNN 过程做个比较和思考。**
>
> * RNN/LSTM 的处理方式：它们是严格按顺序（时间步）处理词的。处理第 t 个词时，需要依赖前一个时间步（t-1）的隐藏状态和当前词的输入。这个隐藏状态本身就包含了之前所有词的信息（但可能会衰减或丢失）。词的位置信息是隐含在处理顺序中的。
> * 训练阶段中，Transformer Encoder/Decoder 处理的是整个序列的“快照”。当你将一个序列（比如长度为 N 的序列）输入到 Transformer 的一层时，它会并行地处理序列中的所有位置。注意力机制会同时计算每个位置与其他位置的相关性。它不是一个词一个词地依赖前一个状态计算当前词的状态。
> * 推理阶段中，Transformer 解码器每一步的输出会作为下一步的输入，直到输出结束符，这个叫<b>自回归生成（autoregressive generation）</b>。比如你用 GPT 生成文本时，它每一步都得等上一步生成完了才继续生成下一个词。所以这时候，它是一步一步来的，有“时间步”概念的。

#### 2.4.2 Self-Attention 机制：“书读百遍，其义自现”

![Transformer Attention](https://drive.google.com/uc?id=1PlpEqXz-uAYOtOy2-5LNv-POgSLJgmU_)

上图展示了 Transformer 架构中的三个 Attention 机制：

1. **编码器中的自注意力机制 (Self-Attention):**
   - 在编码器的每个层中，都会有一个自注意力机制。这一部分的作用是让编码器能够关注输入序列中不同位置的词之间的关系。每个输入词的表示会根据其他所有输入词的表示进行加权，从而生成新的表示。
2. **解码器中的自注意力机制 (Self-Attention):**
   - 在解码器的每个层中，也有一个自注意力机制。这一部分的作用是让解码器在生成输出时，能够关注之前生成的输出词（即解码器的输入）之间的关系。解码器的自注意力机制通常会使用掩码（masking）来确保当前生成的词只能依赖于之前的词，防止信息泄露。
3. **编码器-解码器注意力机制 (Encoder-Decoder Attention)/交叉注意力机制（Cross-Attention）:**
   - 这是 Transformer 架构中第三个注意力机制，位于解码器中。它的作用是让解码器能够关注编码器的输出。通过计算解码器当前状态与编码器所有隐藏状态之间的注意力分数，解码器可以有效地利用编码器提取的上下文信息来生成输出。

> 🤔 **自注意力机制（self-Attention）和前面提到的注意力机制有什么区别？**
>
> 自注意力机制（self-Attention）是注意力机制的一种应用，但特别之处在于，它允许模型在处理序列中的某个元素时，去关注这个<b>同一序列</b>中的其他元素。在自注意力中，用来计算注意力的<b>查询 (Query, Q)、键 (Key, K) 和 值 (Value, V)</b>全部来自于同一个序列。在 Encoder 中，自注意力让模型理解输入序列中词语之间的内部关系（比如代词“它”指的是前面的哪个名词）。在 Decoder 中，自注意力（通常是带掩码的）让模型理解已生成的目标序列内部词语之间的关系。
>
> 注意力机制就像你在翻译时，看着目标语言的句子（解码器），然后回头去源语言的句子（编码器）中找对应的词或短语来帮助翻译。你在两个不同的句子之间建立联系。自注意力机制就像你正在阅读同一篇文章的某个句子，为了理解这个句子中的一个词，你需要回顾一下这同一篇文章/句子中前面的词或后面的词来帮助理解上下文。你在同一个句子内部建立联系。
> <table><tr>
> <td><img src="https://drive.google.com/uc?id=1jShz3F1bm7gOOkhCnw2wRzpwCvL4PY2Z"></td>
> <td><img src="https://drive.google.com/uc?id=1bsQziTjm13WROSl5uQGKSEDbBauvMwJP"></td>
> </tr></table>



在 Transformer 中，注意力机制通过计算查询 (Query) 与键 (Key) 的相似度，然后据此对值 (Value) 进行加权求和来实现。查询（Query）、键（Key）和值（Value）是通过对输入嵌入进行线性变换得到的。具体的步骤如下：

1. 输入嵌入：首先，输入序列会被转换为嵌入向量。假设输入序列的嵌入表示为 $X$，其中 $X$ 是一个形状为 $(n, d)$ 的矩阵，$n$ 是序列的长度（词的数量），而 $d$ 是嵌入的维度。

2. 线性变换：为了得到查询（Q）、键（K）和值（V），我们需要对输入嵌入 $X$ 进行线性变换。具体来说，我们会使用三个不同的权重矩阵来生成 Q、K 和 V：
   - **查询（Q）**：$Q = X W_Q$
   - **键（K）**：$K = X W_K$
   - **值（V）**：$V = X W_V$

其中，$W_Q$、$W_K$ 和 $W_V$ 是可学习的权重矩阵，通常它们的形状为 $(d, d_k)$，$(d, d_k)$ 和 $(d, d_v)$，分别对应查询、键和值的维度。这里 $d_k$ 和 $d_v$ 是键和值的维度，通常可以与嵌入维度 $d$ 相同或不同。

$Q$、$K$、$V$ 是通过对输入嵌入进行线性变换得到的，分别对应于查询、键和值。通过这些向量，模型能够计算输入序列中各个元素之间的关系，从而实现自注意力机制。这种设计使得模型能够动态地关注输入序列的不同部分，提取出有用的上下文信息。
  $$
  Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V
  $$
缩放因子 $\frac{1}{\sqrt{d_k}}$ 用于防止内积过大导致的梯度消失

<img src="https://drive.google.com/uc?id=1hU1hkmBBVpXL0szv08crLVbiz-84PCp4" width="800px">

图源：[Sebastian Raschka](https://sebastianraschka.com/blog/2023/self-attention-from-scratch.html)

Transformer 解码器中的**交叉注意力 (Cross-Attention)** 机制是连接编码器和解码器的关键，它允许解码器在生成目标序列的当前部分时，通过将自己的表示作为**查询 (Query, Q)**，去关注（加权）编码器对**源序列**处理后得到的表示（作为**键 Key, K 和 值 Value, V**），从而有效地从源序列中提取最相关的上下文信息来指导生成。

<img src="https://drive.google.com/uc?id=1seiemdHCMVXWJe-ndJ6z17A15Lf5jOwv" width="800px">

图源：[Sebastian Raschka](https://sebastianraschka.com/blog/2023/self-attention-from-scratch.html)

> 🤔 <strong>怎么理解架构中的交叉注意力机制模块？</strong>
>
> 我们用一个尽可能直观的比喻来解释。想象一下你是一位同声传译员，正在把一段中文演讲翻译成英文。
> *  **编码器 (Encoder) - 理解中文演讲者：**
>    *   首先，你（编码器）仔细听完了整段中文演讲（输入中文序列）。
>    *   你不仅仅是记住了每个中文词，更重要的是理解了每个词在整个演讲中的含义、作用以及和其他词的关系。你把这些深刻理解后的信息，为每个中文词（或 Token）都生成了一个“**浓缩笔记**”（中文 Token Embedding，也就是 K 和 V ）。这些笔记包含了这个词的丰富信息和上下文。现在你有一整套关于中文演讲的深度理解笔记。
> *  **解码器 (Decoder) - 开始说英文，边说边看笔记：**
>    *   现在轮到你（解码器）开始说英文了。你不是一口气说完，而是一个词一个词地往外蹦（生成英文序列）。
>    *   假设你已经说了一部分英文，比如 "I love..." （当前已生成的英文序列）。
>    *   现在，你需要决定下一个英文词应该是什么？ 光看你已经说出的 "I love..." 可能不够，因为原文可能有不同宾语。你需要回头看看你的中文深度理解笔记！
> *  **交叉注意力机制 (Cross-Attention) - 关键的“看笔记”环节：**
>    *   **你的“问题”(Query - Q)：** 你当前的状态是“我已经说了'I love...'，接下来该说啥呢？” 这个问题或者说你当前的关注点，是基于你正在生成的英文内容的（来自于英文 Token Embedding）。这就是 **Q**，它代表了你**当前生成英文的需求**。
>    *   **笔记的“索引/标签”(Keys - K)：** 你的中文深度理解笔记里有很多条，每一条（对应一个中文 Token）都有一个“标签”，能大致表明这条笔记是关于什么的（比如关于“我”，关于“爱”，关于“你”）。这就是 **K**，它来自于**编码器对中文的理解**。
>    *   **笔记的“详细内容”(Values - V)：** 每条笔记的“标签”背后，是这条笔记的详细、丰富的内容，包含了那个中文词的深层含义和上下文信息。这就是 **V**，也来自于**编码器对中文的理解**。
>    *   **“看笔记”的过程：**
>        *  **匹配问题和标签 (Q vs K)：** 你拿着你当前的需求 Q（“我刚说了'I love'，需要找宾语”），去和你所有的中文笔记标签 K（“我”、“爱”、“你”等）进行比对。你会发现，标签 K 为“你”的那条笔记，和你当前找宾语的需求 Q 最相关！而标签 K 为“我”或“爱”的笔记，相关性就低很多。这个匹配过程会给每条中文笔记打一个 **“相关度分数”（Attention Weights）**。
>        *  **提取相关内容 (Weighted Sum of V)：** 你根据上一步打出的“相关度分数”，**有侧重地去阅读中文笔记的详细内容 V**。对于“你”那条笔记（相关度最高），你会重点吸收它的信息；对于“我”、“爱”的笔记（相关度低），你可能就扫一眼甚至忽略。你把这些**按相关度加权**后的中文笔记内容 V 汇总起来，形成了一个**针对你当前英文需求 Q 的、“定制化”的中文信息精华**。
> *  **为什么输出不是中文？—— 解码器的最后一步**
>    *   **关键点：** 交叉注意力机制的输出——那个中文信息精华（V 的加权求和），**并不是直接作为下一个英文词输出！** 它的作用是作为极其重要的参考资料或上下文线索，被喂给解码器接下来的层（通常是一个前馈神经网络）。
>    *   解码器会综合考虑三个方面的信息：
>        * 它自己内部的“英文语法知识库”（模型参数）。
>        * 它刚刚通过**自注意力机制(Self-Attention)** 理解到的、自己已经说出的英文部分的上下文（比如 "I love..." 的语法结构）。
>        * 刚刚通过**交叉注意力机制(Cross-Attention)** 递交过来的、高度相关的“中文信息精华”。
>    *   **最终决策：** 综合了这三方面信息后，解码器才能做出最合理的决策，**预测出下一个最可能出现的英文 Token**，比如 "you"。

#### 2.4.3 多头注意力机制 (Multi-Head Attention)

为了增强模型的表达能力，Transformer 使用了多头注意力机制，即同时使用多组不同的注意力机制。每个头关注序列的不同方面:

- 有的头可能关注相邻词之间的关系
- 有的头可能关注语法结构
- 有的头可能关注语义关联

在 Transformer 的论文中，使用了 8 组注意力头，每个头都可以关注到序列中不同的方面，从而提高模型的表达能力。

![Multi-head](https://drive.google.com/uc?id=1LMHWMjmRgLirZ2j_rbydoaJePShUJDtF)

上图：多头注意力机制的矩阵运算

![Example](https://drive.google.com/uc?id=1K8LP9djH87OVeknmZpU_Tw4C73xycDwX)

图中的绿色和橙色线条分别表示2组不同的attentin heads。当我们编码单词 `it` 时，其中一个 attention head （橙色注意力头）最关注的是 `the animal` ，另外一个绿色 attention head 关注的是 `tired`。因此在某种意义上，`it` 在模型中的表示，融合了 `animal` 和 `tired` 的部分表达。

#### 2.4.4 残差链接（Residual Connection） & 层标准化（Layer-normalization）

在 单层 Encoder 中，我们注意到有 2 个 Add & Normalize 层 （残差链接和层标准化）。

残差链接是指在神经网络的某一层输出与该层输入之间建立直接的连接。具体来说，对于某一层的输出 $ y $，残差链接的形式可以表示为：

$$ y = F(x) + x $$

其中：
- $F(x)$ 可以是自注意力层的输出或前馈网络的输出。
- $x$ 是该层的输入。

残差链接的作用：
- 缓解梯度消失问题：在深层神经网络中，随着层数的增加，梯度在反向传播时可能会逐渐减小，导致梯度消失。这是因为在每一层中，激活函数（如sigmoid或tanh）可能会导致梯度变得非常小，尤其是在网络较深时。残差链接通过引入直接路径，使得梯度可以在反向传播时不经过多个层的变换，从而保持较大的值，从而缓解了梯度消失的问题。
- 促进信息流动：残差链接允许网络在学习过程中保留输入信息。即使在某些层中，网络学习到的特征不够理想，输入信息仍然可以通过残差链接传递到后续层。这种设计使得网络能够更好地学习到有用的特征。
- 加速收敛：通过引入残差链接，网络可以更快地收敛到较低的损失值。研究表明，残差网络（ResNet）在训练时通常比没有残差链接的网络收敛更快。

在Transformer中，残差链接通常与层归一化（Layer Normalization）结合使用，层归一化有助于稳定训练过程，缓解梯度消失的问题。通过对每一层的输出进行归一化，模型能够更好地控制激活值的范围，从而进一步减少梯度消失的风险。

$$ \text{Output} = \text{LayerNorm}(F(x) + x) $$

<img src="https://drive.google.com/uc?id=1zLW1xMpOllwGwg3OUMjWHaGcsMQbcIjG" width=600px>

图中的 $Z$ 对应的是上述公式中的 $F(x)$。

> 🤔 <strong>为什么用 Layer Normalization 而不是更常见的 Batch Normalization？</strong>
>
> Batch Normalization 对一批样本的同一维度特征进行归一化，抹杀了不同特征之间的大小关系，但是保留了不同样本间的大小关系，适用于图像处理等需要处理大量数据的情况，因为它依赖于批次中的样本数量来估计均值和方差；Layer Normalization 对单个样本的所有维度特征进行归一化，抹杀了不同样本间的大小关系，但是保留了一个样本内不同特征之间的大小关系，适用于自然语言处理等任务，特别是当处理的数据长度不同时，因为它不依赖于批次大小，而是依赖于单个样本的维度。


#### 2.4.5 FFN

在Transformer架构中，前馈网络层（Feed Forward Network）位于自注意力层之后，主要作用是对自注意力层的输出进行进一步的处理和转换，通常由两个线性变换和一个激活函数组成，具体结构如下：

$$ \text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2 $$

其中：
- $ x $ 是输入（自注意力层的输出）
- $ W_1, b_1 $ 和 $ W_2, b_2 $ 是可学习的权重矩阵和偏置项
- $ \max(0, \cdot) $ 是 ReLU 激活函数

**主要作用**
- 非线性变换：自注意力机制主要关注序列中各个元素之间的关系，但它本身是一个线性操作。前馈网络层通过引入非线性激活函数（如ReLU），能够对自注意力层的输出进行非线性变换，从而增强模型的表达能力。
- 特征提取：前馈网络层可以看作是一个特征提取器，它能够从自注意力层的输出中提取更高层次的特征。通过两个线性变换和激活函数，前馈网络层能够捕捉到更复杂的模式和关系。
- 维度扩展和压缩：前馈网络层通常会将输入向量的维度进行扩展，然后再压缩回原来的维度。这种扩展和压缩的过程有助于模型学习到更丰富的特征表示。

<img src="https://drive.google.com/uc?id=1Wfv-ybsmoIItHRnDWOsW-9ml13qOM-s4" width=600px>

#### 2.4.6 线性层和 Softmax

Decoder 最终的输出是一个向量，其中每个元素是浮点数。我们怎么把这个向量转换为单词呢？这是线性层和softmax完成的。

线性层就是一个普通的全连接神经网络，可以把解码器输出的向量，映射到一个更大的向量，这个向量称为 logits 向量：假设我们的模型有 10000 个英语单词（模型的输出词汇表），此 logits 向量便会有 10000 个数字，每个数表示一个单词的分数。

然后，Softmax 层会把这些分数转换为概率（把所有的分数转换为正数，并且加起来等于 1）。然后选择最高概率的那个数字对应的词，就是这个时间步的输出单词。

![linear-softmax](https://drive.google.com/uc?id=1V7IUllY8xeajsemsqazIQq9_YsSWtT4n)

>
> <strong>🌟 再回头理解 Q、K、V: </strong> Transformer 中的 Q（Query）、K（Key）、V（Value）在不同任务中的含义有什么不同。
>
> * 机器翻译（如英译德）
>    * Encoder 输入端：Q、K、V 都指代英文的 embedding，执行的是 self-attention。
>    * Decoder 输入端：Q、K、V 都指代德文的 embedding，执行的是 masked self-attention。
>    * Encoder 输出和 Decoder 输出交互时：执行 cross-attention，此时 Q 指代德文的 embedding，K、V 指代英文的 embedding。
>
> * 车道线检测（CLRNet）
>    * Q：指代车道线的特征。
>    * K、V：指代图像的 feature map，用来计算车道线特征对图像全局特征的注意力情况。
>
> * 图像描述（Image Captioning）
>    * Q：指代 LSTM 的输出。
>    * K、V：指代图像的特征图（feature map）。
>
> * 目标检测（DETR）
>    * Q：指代 object query，是模型权重的一部分，即可学习的参数。
>    * K、V：同样指代图像的 feature map。
>
> 总结：**Q 一般与具体任务相关，而 K、V 是 Q 想要关注的物理量**。Transformer 结构能够在不同领域中使用，体现了它的通用性。
>

### 2.5 BERT / GPT

当今的 LLMs，Transformer 架构是一个主流的选择。但是不一定都是 Transformer 完整架构的实现。下表列出了一些常见的 Transformer 模型。

| 架构类型 | 模型名称 | 发布时间 | 开发机构 | 参数规模 | 主要特点 | 典型应用场景 |
|---------|---------|----------|----------|----------|----------|-------------|
| **Encoder-Only** | BERT | 2018 | Google | 340M | • 双向编码<br>• MLM预训练<br>• 强大的语义理解能力 | • 文本分类<br>• 命名实体识别<br>• 问答系统 |
| | RoBERTa | 2019 | Facebook | 355M | • BERT的改进版<br>• 移除NSP任务<br>• 动态掩码策略 | • 文本分类<br>• 序列标注<br>• 情感分析 |
| | DeBERTa | 2020 | Microsoft | 750M | • 解耦注意力机制<br>• 增强的位置编码<br>• 更好的性能 | • 自然语言理解<br>• 文本分析<br>• 情感分类 |
| **Decoder-Only** | GPT-3 | 2020 | OpenAI | 175B | • 强大的生成能力<br>• 少样本学习<br>• 任务普适性 | • 文本生成<br>• 对话系统<br>• 代码生成 |
| | LLaMA | 2023 | Meta | 7B-70B | • 开源基础模型<br>• 高效架构<br>• 多种规模可选 | • 通用文本生成<br>• 对话<br>• 推理 |
| | Claude | 2023 | Anthropic | 未公开 | • 强调安全性<br>• 长文本处理<br>• 指令跟随能力强 | • 对话助手<br>• 文档分析<br>• 代码辅助 |
| **Encoder-Decoder** | T5 | 2019 | Google | 220M-11B | • 统一文本任务框架<br>• 强大的迁移能力<br>• 灵活的任务适应 | • 文本摘要<br>• 机器翻译<br>• 问答系统 |
| | BART | 2019 | Facebook | 140M | • 序列到序列预训练<br>• 噪声去除任务<br>• 生成质量高 | • 文本生成<br>• 机器翻译<br>• 摘要生成 |
| | mT5 | 2021 | Google | 300M-13B | • 多语言版T5<br>• 支持101种语言<br>• 跨语言迁移能力 | • 多语言翻译<br>• 跨语言任务<br>• 文本生成 |

注意事项：
1. 参数规模可能随版本更新而变化
2. 部分模型可能有多个版本，表格中列出最具代表性的版本
3. 应用场景仅列举最典型的使用情况
4. 特点描述侧重模型的关键创新和主要优势


其中，BERT 和 GPT 是基于 Transformer 架构的最成功的两个代表性模型系列，但它们在具体实现和预训练目标上有所不同，导致了能力的侧重不同。
- **BERT (Bidirectional Encoder Representations from Transformers, Devlin et al., 2019):**
   - **架构:** 主要使用 Transformer 的 **编码器 (Encoder)** 部分。
   - **特点:** 通过 Masked Language Model (MLM) 预训练任务，可以**同时利用左右两侧的上下文**信息来预测被遮盖的词 (双向性)。
   - **擅长:** 理解任务 (NLU)，如文本分类、命名实体识别、问答等，因为它能深入理解词在完整语境中的含义。
- **GPT (Generative Pre-trained Transformer, Radford et al., 2018 onwards):**
   - **架构:** 主要使用 Transformer 的 **解码器 (Decoder)** 部分（稍作修改，去除了编码器-解码器注意力）。
   - **特点:** 通过标准的自回归语言模型任务（预测下一个词）进行预训练。在预测当前词时，**只能看到左侧的上下文** (单向性或自回归性)。
   - **擅长:** 生成任务 (NLG)，如文本生成、对话系统、摘要、翻译等，因为它天然适合按顺序生成文本。

<img src="https://drive.google.com/uc?id=1g_BhPLBQEMPEMzFmK4Jg-Ro-Cnn7QxTb" width=600px>

**优势与局限性**

| 架构类型     | 优势           | 局限性           |
| ------------ | -------------- | ---------------- |
| Encoder-Only | 双向上下文理解 | 不适合生成任务   |
|              | 并行处理效率高 | 需要固定长度输入 |
|              | 适合理解类任务 | 计算资源消耗大   |
| Decoder-Only | 自然的生成能力 | 生成过程串行     |
|              | 灵活的输出长度 | 无法利用后文信息 |
|              | 适合开放式任务 | 推理速度较慢     |

> <strong>🤔 Transformer 出来后，Encoder-only 模型曾经几乎一统天下，为什么后来 Decoder-only 能成功逆袭？</strong>
>
> Encoder-only 模型的训练模式大多采用了Masked LM（随机挡住句子中一些词，然后让模型给还原出来)，另外一种叫Next Sentence Prediction(让模型判断句子对是否前后相邻关系)，简单理解就是让模型做完形填空和判断句子关联性；Decoder-only 模型的生成模式为自回归，mask方式为后面的token都给设置成负无穷，也就表示不会被计算到，所以可以理解为，第一轮开局一个字，后面全靠编，打个比方就是写大作文。
>
> 语言考试最难的题肯定不是完型填空，那必然是憋作文啊，所以 Decoder-only 模型的训练方式要比 Encoder-only 模型更难，消耗资源也多，反过来讲一个成天写作文的学生，语言能力大概率是比成天做完形填空的学生好的。

之前的模型需要针对不同任务设计特定架构或进行大量任务特定的训练，BERT/GPT带来的范式（更具体的内容会在角度三讲解）：
1.  **预训练-微调范式:** 在大规模无标签数据上预训练一个通用的 Transformer 模型，然后在特定下游任务的小型有标签数据上进行微调 (Fine-tuning)，极大提升了模型性能和效率。
2.  **强大的基础模型:** BERT 和 GPT 及其后续变体 (RoBERTa, ALBERT, GPT-2, GPT-3, GPT-4 等) 成为了众多 NLP 应用的基础。

<div class="alert alert-info">
<b>扩展阅读（本课程 2.4 & 2.5 节借鉴了大量 Jay Alammar 这两篇文章的内容，推荐大家认真阅读）：</b>
<ul>
<li><a href="https://jalammar.github.io/illustrated-transformer/">图解 Transformer</a></li>
<li><a href="https://jalammar.github.io/illustrated-gpt2/">图解 GPT</a></li>
</ul>
</div>

### 2.6 走出 Transformer：下一代 AI 架构的三大前沿探索

#### **思考：为什么统治 AI 界的 Transformer 还需要被超越？**

* 算力爆炸：工作量按平方级（$O(N^2)$）暴增。
* 逐字蹦词：写文章时像老式打字机一样从左到右一个字一个字往外蹦，既慢又缺乏全局规划。

为了解决这两大底层痛点，过去两年科学家们从**人类阅读、记忆与创作**的直觉出发，开辟了一些极具代表性的新路线。

#### **路线一：选择性状态空间模型（Mamba）**

**核心思想类比：录音机 vs. 聪明速记员**

为什么人类读完一本厚书，脑袋不会像电脑内存一样被撑爆？因为人类**懂得选择性遗忘**！

| 维度 | 传统 Transformer（全程录音机 / 便利贴狂魔） | Mamba 选择性状态空间（聪明速记员） |
| :--- | :--- | :--- |
| **工作方式** | 不管是核心机密还是“哈哈、那个、今天天气不错”等废话，全部一视同仁贴上便利贴保存。 | 手里只有**一本固定厚度的活页笔记本**（固定大小的状态空间）。 |
| **核心创新** | 无筛选全量缓存 | **选择性阀门（Selective Scan）**：看到关键信息（人名、金额、转折）立刻打开阀门写入笔记本；看到寒暄废话直接关门略过。 |
| **内存消耗** | 随文章长度线性暴涨（读得越长，桌子越放不下） | **永远恒定**（无论读 1 千字还是 100 万字，只占那一小本笔记的空间！） |

**直观原理示意图**
```
输入文本流：  "呃..." ──▶ "今天天气不错" ──▶ "【合同金额：500万】" ──▶ "哈哈好的" ──▶ "【违约金：20%】"
                 │              │                     │                    │                 │
Mamba选择阀门： [关闭 0.05]    [关闭 0.10]          [全开 0.98!]          [关闭 0.08]       [全开 0.95!]
                 │              │                     ▼                    │                 ▼
固定大小笔记本： [  保持原样  ] ──▶ [  保持原样  ] ──▶ [写入：金额500万] ──▶ [ 保持原样 ] ──▶ [金额500万 + 违约20%]
```

**Mamba 的创新可以用三个层次来理解：**
1. SSM： 通过内部状态来处理序列，像一个不断更新的记忆系统。
2. Selective： 让状态更新方式由输入决定，学会选择性记忆，而不是机械地处理每个 token。
3. Selective Scan： 通过专门的并行算法，让这种动态记忆机制仍然能够高效计算。
还有一个技术细节值得纠正：Mamba 并不是让 A、B、C、D 全部随输入变化。经典 Mamba 中，B、C 和步长参数 Δ 是输入相关的，基础参数 A 和 D 则保持固定；离散化后的状态更新参数会随 Δ 变化。

简单理解：Transformer 像一个随时可以翻阅全部历史记录的人；Mamba 像一个善于做笔记的人，边读边判断什么值得记住，并把记忆不断更新。

这就是 Mamba 最核心的思想：让模型具备选择性记忆能力，同时保持线性时间复杂度。

#### **路线二：扩散语言模型（dLLMs）与世界模型（JEPA）**

Mamba 在改进 AI **“怎么记东西”**，本路线则直接颠覆了 AI **“怎么说话、怎么理解世界”**：

##### **文本扩散大模型（Diffusion LLMs，如 LLaDA / Mercury 2）**

* **传统自回归模型（GPT / DeepSeek 等）**：像一个**不能用退格键的打字员**，必须按顺序从第 1 个字猜到最后 1 个字。写一篇 500 字的文章，必须串行跑 500 步；而且如果开头第一句话说劈叉了，后面只能硬着头皮圆谎。
* **文本扩散大模型（dLLMs）**：把 AI 画图（Midjourney / Stable Diffusion）的“去噪显影”思想搬到了文字上！
  * **第 0 步**：先在纸上摆好整句话长度的“空白迷雾块”（`[░]`）；
  * **第 1 步**：全局同时动笔，先把整句话的**核心骨架词**（主语、谓语、转折词）显影出来；
  * **第 2~3 步**：全篇同时填补形容词、细节与标点，如果发现前后不搭还可以当场修改！
  * **震撼效果**：不需要一个字一个字排队，**生成速度直接飙升 5~10 倍（每秒可输出上千字）**，而且在写代码、填字游戏、全局规划上天生具有“上帝视角”。

##### **联合嵌入预测架构（JEPA）**

图灵奖得主 Yann LeCun（杨立昆）提出了一个发人深省的问题：
> **“当你看到一辆汽车驶向十字路口时，你的大脑会去预测下一秒车轮旁边飞起的每一粒灰尘、路边大树上每一片树叶晃动的像素位置吗？”**

当然不会！人类之所以拥有强大的物理常识，恰恰是因为**我们的大脑会自动忽略杂乱无章的底层细节（像素噪声），直接在高维的“抽象概念空间”里做预测**（比如只预测：“车会减速停下”或“车会右转”）。

* **传统生成式模型（Pixel/Token 级预测）**：强迫 AI 把画面里的每一片树叶、每一朵水花都一五一十画出来，结果 90% 的算力都浪费在毫无意义的背景杂波上，反而经常搞错基本的物理规律。
* **JEPA（联合嵌入预测架构）**：彻底放弃“画全每一个像素”，而是先把画面提炼成**核心语义特征（比如物体的位置、速度、意图）**，直接在抽象脑海里预测未来的状态。



### ✅ 小结：架构的演进回顾

|年代 | 阶段 | 架构 | 主要特点 |
|---------|------|-----------|----------|
| 1950s | ANN| 	前馈网络	| 无序列建模能力|
| 1990 | RNN	| 循环结构	| 有记忆，难训练|
| 2014 | Attention	| 注意力机制	| 任意位置交互，信息直达|
| 2017 | Transformer	| 堆叠 Attention	| 并行、高表达力|
| 2018 | BERT/GPT| 	编码/解码范式	| 理解/生成分流|
| 2023～ | Mamba、JEPA 等| 	轻量、新结构	| 高效替代 Transformer|

> 模型架构从无法处理序列的 ANN，发展到能够处理序列但受限于梯度和并行性的 RNN，再到通过 Attention 机制突破瓶颈，最终演进到以 Self-Attention 为核心、并行高效且能捕捉长距离依赖的 Transformer 架构。BERT 和 GPT 将 Transformer 的威力发挥到极致。而 Mamba 等新兴架构则在尝试解决 Transformer 在超长序列上的局限性，探索新的可能性。架构的每一次革新，都极大地提升了模型处理和理解复杂信息的能力。

## 3. 范式的革命 / 任务的大一统 —— 从特定任务到通用智能

除了表示和架构，我们如何 **训练 (Train)** 和 **使用 (Use)** 模型来解决各种自然语言任务，即所谓的 **范式 (Paradigm)**，也经历了深刻的变革。这种变革趋势指向了 **任务的大一统 (Task Unification)**，即用一个（或少数几个）强大的通用模型来解决多种多样的任务。

本章节通过具体的自然语言处理（NLP）任务，演示机器学习解决问题范式的演变过程。我们将依次探讨基于规则、统计机器学习、深度学习、预训练+微调以及提示学习这五种范式，并观察它们是如何逐步走向一种更统一的任务解决框架。我们从为每个任务设计特定规则/特征/模型，逐渐过渡到使用强大的通用模型，通过少量调整（微调）或甚至仅通过自然语言指令（提示）来适应不同任务。

### 3.1 基于规则的方法 (Rule-based Approaches)

早期的 NLP 系统严重依赖**语言学家手工编写的规则**。例如，语法分析器可能基于乔姆斯基的形式文法，情感分析可能依赖包含正面/负面词汇的词典和规则（如“not good”表示负面）。

<img src="https://drive.google.com/uc?id=16Plh2wzR3CGXbK4JrxJj3ctSY0iCzeLT" width=800px>

**问题:**
1.  **人力成本高昂:** 需要语言学专家耗费大量时间和精力来设计和维护规则。
2.  **覆盖范围有限 (Low Coverage):** 语言现象复杂多变，规则很难覆盖所有情况，尤其是在口语、网络用语等非规范场景下。
3.  **泛化能力差 (Poor Generalization):** 规则通常很脆弱 (brittle)，难以适应新的领域或语言风格。遇到未覆盖的情况时，系统表现会急剧下降。
4.  **难以处理歧义:** 复杂的语义歧义很难用明确的规则来解决。

### 3.2 统计学习方法 (Statistical Learning Methods)

随着计算能力和数据量的增长，研究范式转向**从数据中自动学习模式**，如朴素贝叶斯 (Naive Bayes)、支持向量机 (SVM)、逻辑回归 (Logistic Regression) 等，<font color="red">**通常结合人工设计的特征**</font>。

🌟 <b>不同任务中常用的特征</b>


任务1：文本分类 (Text Classification)

| 特征类型                 | 描述                                                                 | 举例/关键技术                                     |
| :----------------------- | :------------------------------------------------------------------- | :------------------------------------------------ |
| 词袋模型 (BoW)       | 忽略词序，统计词语出现频率。                                         | Tokenization, Stopwords Removal, Stemming/Lemmatization |
| TF-IDF              | 在BoW基础上加权，突出重要但不常见的词。                               | `TF * log(N / (df + 1))`                         |
| N-grams              | 考虑连续的N个词，捕捉局部语序信息。                                  | Bigrams ("机器 学习"), Trigrams ("统计 机器 学习")      |
| 词性 (POS) 特征     | 统计名词、动词、形容词等的频率或分布。                               | 使用POS Tagger获取词性标签                        |
| 句法特征           | 基于句法树分析，提取依存关系、短语结构等。                           | Dependency Parsing, Constituency Parsing          |
| 主题模型特征         | 使用LDA等模型提取文档的主题分布向量。                                | Latent Dirichlet Allocation (LDA)                  |
| 文本统计特征        | 文档长度、平均句长、大写字母比例、特定标点符号数等。                   | 直接计算统计值                                    |

任务2：命名实体识别 (Named Entity Recognition, NER)
    
| 特征类型                | 描述                                          | 举例/关键技术                                          |
| ----------------------- | --------------------------------------------- | ------------------------------------------------------- |
| 词本身特征              | 当前词的小写、词干、词形还原形式。               | `word.lower()`, `stem(word)`, `lemmatize(word)`          |
| 词形特征 (Word Shape)   | 捕捉词的构成模式（大写、数字、标点等），忽略具体字母。 | `"Xxxx"`, `"ddd"`, `"xx-xx"`, `"AllCaps"`              |
| 词性 (POS) 特征         | 当前词及其上下文词的POS标签。                   | `NLTK`, `SpaCy POS Taggers`                              |
| 词缀特征                | 当前词的前缀、后缀 (如最后2、3个字母)。          | `word[-3:]`, `word[:2]`                                 |
| 上下文特征              | 前后N个词的词本身、词形、POS等特征。            | `Window size of +/- N tokens`                           |
| 词典/地名录 (Gazetteer) | 当前词或短语是否在预定义的实体列表中。             | 城市名列表, 常见人名列表 (`Boolean feature`)            |
| 词聚类特征              | 词所属的预训练聚类ID (如Brown Cluster)。       | `Brown Clustering`, `(Early Word Embeddings)`            |

    
任务3：图像分类 (Image Classification)

| 特征类型                   | 描述                                                               | 关键技术/方法                                                  |
| :------------------------- | :----------------------------------------------------------------- | :------------------------------------------------------------- |
| 全局特征: 颜色直方图       | 统计图像中不同颜色（或灰度）的像素数量。                           | `RGB`, `HSV color spaces`                                      |
| 全局特征: 纹理 (GLCM)      | 灰度共生矩阵，描述像素灰度空间关系，计算对比度、能量等。           | `GLCM` (Gray-Level Co-occurrence Matrix)                       |
| 全局特征: 纹理 (LBP)       | 局部二值模式，描述像素与邻域关系，对光照鲁棒。                     | `LBP` (Local Binary Patterns)                                  |
| 全局特征: 纹理 (Gabor)     | 使用Gabor滤波器组提取多尺度、多方向的纹理信息。                    | `Gabor Filters`                                                |
| 全局特征: 形状             | （需先分割）描述物体轮廓或区域的几何属性。                         | `Moments`, `Fourier Descriptors`                               |
| 局部特征: SIFT/SURF        | 检测尺度、旋转不变的关键点，并生成描述符。                           | `SIFT` (Scale-Invariant Feature Transform), `SURF` (Speeded Up Robust Features) |
| 局部特征: HOG              | 统计图像局部区域的梯度方向分布。                                   | `HOG` (Histogram of Oriented Gradients)                        |
| 视觉词袋模型 (BoVW)        | 模仿文本BoW：提取局部特征 -> 聚类成视觉词典 -> 统计词频直方图。        | `K-Means clustering` on `SIFT`/`SURF` descriptors              |
| BoVW + 空间金字塔 (SPM)    | 在BoVW基础上，加入不同空间尺度的块划分，保留粗略位置信息。           | `SPM` (Spatial Pyramid Matching) (builds on `BoVW`)            |


任务4：目标检测 (Object Detection)
    
| 特征类型           | 描述                                                               | 关键技术/方法                                |
| :----------------- | :----------------------------------------------------------------- | :------------------------------------------- |
| Haar-like 特征 | 基于矩形区域像素和的差值，对边缘、线段敏感，计算速度快（配合积分图）。 | Viola-Jones Detector, Integral Image       |
| HOG 特征       | 在滑动窗口内计算梯度方向直方图，对行人形变和光照有鲁棒性。           | Dalal-Triggs Detector (Pedestrian Detection) |
| SIFT/SURF + BoVW | 在窗口内提取局部特征，用BoVW表示窗口内容，然后分类。               | Less common as primary detector feature than Haar/HOG in early days |


**相比规则方法，统计学习的优势包括:**
1.  **自动化学习:** 模型从标注数据中自动学习规律，减少了手工编写规则的工作量。
2.  **更好的泛化:** 基于统计模式，通常比硬编码的规则具有更好的泛化能力。
3.  **数据驱动:** 性能很大程度上取决于数据的质量和数量。

**统计学习的局限性:**
- **特征工程 (Feature Engineering):** 性能高度依赖于人工设计的特征质量。设计好的特征需要领域知识和经验，仍然是劳动密集型的过程。
- **无法捕捉深层语义:** 基于浅层特征（如词袋），难以捕捉复杂的语义关系和长距离依赖。
- **任务特定:** 通常需要为每个任务单独设计特征和训练模型。

### 3.3 深度学习 (Deep Learning, Pre-Transformer Era)

随着深度学习的兴起，神经网络（尤其是 RNN, LSTM, CNN）开始被广泛应用于 NLP 任务。模型可以**自动从原始文本中学习特征表示**，避免了繁琐的人工特征工程。Word2Vec 等预训练词嵌入也常被用作模型的输入层。
1.  **自动特征学习:** 神经网络能够自动学习数据中的层次化特征。
2.  **更强的表示能力:** 能够捕捉更复杂的模式和语义关系（尤其是 RNN/LSTM 对序列的处理）。
3.  **端到端学习 (End-to-End):** 可以直接从原始输入学习到最终输出，简化了流程。

<div class="alert alert-info">
    🌟 <b>我们尝试着来分析一下这个训练过程。</b>
    
1. **训练有进展，但出现潜在问题：** 模型在训练集上的表现持续提升，但在验证集上出现了性能瓶颈与过早退化的迹象。
2. **性能指标变化：**
    * **训练损失 (Train Loss)** 从 0.691 持续下降到 0.645。
    * **训练准确率 (Train Acc)** 从 52.17% 提升到 63.25%。
    * **验证损失 (Val. Loss)** 在 Epoch 1 达到 0.674 后未能继续下降，在 Epoch 2 微升至 0.682，并在 Epoch 3 大幅上升至 0.806。
    * **验证准确率 (Val. Acc)** 在 Epoch 1 达到 58.60% 后出现回落（Epoch 2 为 54.23%，Epoch 3 为 57.24%）。
3. **明显的过拟合迹象：** 最关键的信号是 **验证损失在 Epoch 1 之后没有改善，反而在 Epoch 2 到 Epoch 3 之间显著上升**。虽然训练损失持续下降，训练准确率持续上升，但验证损失的飙升和验证准确率的回落表明模型在验证集上的泛化能力（从损失函数的角度看）已达到瓶颈，并开始出现明显的过拟合现象。模型可能在学习训练数据中一些对验证集优化损失函数无益（甚至有害）的特定模式。
4. **最佳模型保存基于验证损失：** 日志仅在 Epoch 1 明确指出 `-> Best model saved based on validation loss.`。因此，虽然 Epoch 2 和 Epoch 3 的训练性能更高，但由于验证损失没有低于 Epoch 1 的 0.674，**Epoch 2 和 Epoch 3 结束时并没有保存新的“最佳”模型**。当前保存的最佳模型是 Epoch 1 结束时的状态。

**建议：**

* 密切监控验证损失。由于验证损失已开始明显上升，应考虑停止训练 (Early Stopping)。
* 验证准确率在 Epoch 1 后也出现了停滞和回落，更应及时停止训练并回退使用 Epoch 1 的模型权重，或调小学习率重新训练。
* 可以考虑增加正则化手段（如 Dropout、权重衰减 L1/L2）或进行数据增强，以提高模型的泛化能力，对抗过拟合。
    
</div>

<div class="alert alert-info">
    🌟 <b>我们再来分析一下这个训练过程。</b>
    
1. **训练进展顺利：** 模型在 5 个 Epoch 的训练过程中表现出持续的改进。
2. **性能指标持续优化：**
    * **训练损失 (Train Loss)** 从 1.6323 稳步下降到 0.8965。
    * **训练准确率 (Train Acc)** 从 39.73% 显著提升到 68.76%。
    * **测试损失 (Test Loss)** 从 1.2908 稳步下降到 0.7975。
    * **测试准确率 (Test Acc)** 从 52.40% 显著提升到 71.99%。
3. **模型泛化能力良好：** 测试集上的损失持续下降，准确率持续上升，并且测试集指标始终优于或贴近训练集指标（差距没有异常扩大），表明模型在学习训练数据的同时，也能较好地泛化到未见过的数据上。目前没有明显的过拟合迹象。
4. **最佳模型持续更新：** 日志显示 `-> Best model saved based on test accuracy` 在每个 Epoch 结束时都出现（依次为 `0.5240`、`0.6137`、`0.6721`、`0.7145`、`0.7199`），这意味着每个 Epoch 都带来了测试准确率的提升，并且模型权重被相应保存。截至 Epoch 05 结束并加载最佳模型进行最终评估，达到的最佳测试准确率为 **71.99%**（最终测试损失为 **0.7975**）。
5. **批次内波动：** 每个 Epoch 内的 Batch Loss 和 Acc 有一定的波动（例如 Epoch 03 中 Batch 100 到 200 准确率从 0.6484 波动至 0.5781，随后在 Batch 300 升至 0.7344），这是小批量梯度下降的正常现象。但整体趋势（Epoch 级别的 Loss 和 Acc）依然稳定向好的方向发展。

**总而言之，这是一个非常积极的训练过程。** 模型在训练集和测试集上都表现出良好的学习效果和性能提升。不过可以注意到，从 Epoch 04 到 Epoch 05，测试集准确率的提升幅度有所放缓（从 71.45% 微升至 71.99%，测试损失从 0.8040 微降至 0.7975）。训练可以继续进行以观察性能是否能进一步提高，同时需要开始留意测试集性能是否即将趋于饱和或出现过拟合迹象（即训练准确率持续升高，但测试准确率停滞或下降）。
    
</div>

**此时期深度学习的局限性:**
- **数据需求:** 对于每个特定任务，仍然需要大量的标注数据才能训练出高性能的模型。
- **训练成本:** 训练深度模型计算成本较高。
- **有限的知识迁移:** 虽然可以使用预训练词嵌入，但模型主体结构通常还是需要为特定任务从头开始训练或进行大量调整，知识迁移能力有限。

### 3.4 预训练 + 微调范式 (Pre-training and Fine-tuning)

这是由 ELMo、BERT、GPT 等模型引领的革命性范式，也是当前 LLM 的基础。它包含两个阶段：
1.  **预训练 (Pre-training):** 在 **大规模无标注文本** 数据上，使用 **自监督学习 (Self-supervised Learning)** 任务（如 Masked Language Model - MLM, Next Sentence Prediction - NSP, Standard Language Model - LM）训练一个通用的、深层的基础模型（通常是 Transformer 架构）。这个过程让模型学习广泛的语言知识、语法结构甚至一些世界知识。
2.  **微调 (Fine-tuning):** 针对 **特定的下游任务**（如情感分析、问答、翻译），使用 **少量该任务的标注数据** 对预训练好的模型进行**进一步训练**。通常只调整模型顶部的少数几层或添加一个小的任务特定层。模型利用预训练阶段学到的通用知识，快速适应新任务。

<img src="https://drive.google.com/uc?id=15hPZxA7P13qU9z0Ae6r43gjuXijQEAsN" width=800px>

**优势:**
1.  **强大的知识迁移:** 将从海量无标注数据中学到的通用语言知识有效迁移到各种下游任务，显著提升性能。
2.  **降低对标注数据的依赖:** 只需要少量任务特定的标注数据即可达到很好的效果。
3.  **任务大一统的开端:** 一个预训练模型可以通过微调适配多种任务，减少了为每个任务设计和训练独立模型的需要。
4.  **催生基础模型 (Foundation Models):** 预训练模型本身成为了一种可重用的基础设施。

以 BERT 为例，以下是实现 BERT 多任务微调的关键步骤和机制：

<img src="https://drive.google.com/uc?id=1op3FCcG5ipkHcqvpFjhe-fv4narlPXFL" width=800px>

图源：[BERT](https://ar5iv.labs.arxiv.org/html/1810.04805)

1.  **共享的BERT骨干（Shared BERT Backbone）:**
    *   使用预训练好的BERT模型（例如 `bert-base-uncased`, `bert-large-chinese` 等）作为基础。
    *   这个BERT模型的主体部分（Transformer层）是所有任务共享的。这意味着不同任务的输入都会经过这些相同的层来提取深层次的、上下文感知的语言表示。

2.  **任务特定的输出层（Task-Specific Heads）:**
    *   对于每一个要处理的下游任务，在共享BERT模型的顶部添加一个或多个专门为该任务设计的输出层。
    *   **示例:**
        *   **文本分类 (Text Classification):** 通常在BERT的 `[CLS]` token对应的输出向量上接一个简单的全连接（Linear）层，然后跟一个Softmax激活函数来预测类别概率。
        *   **序列标注 (Sequence Labeling, 如NER):** 在BERT每个输入token对应的输出向量上接一个全连接层，然后可能跟一个Softmax（用于独立分类每个token）或一个条件随机场（CRF）层（用于考虑标签之间的依赖关系）。
        *   **句子对关系判断 (Sentence Pair Classification, 如NLI):** 类似文本分类，通常使用 `[CLS]` token的输出，后接全连接层和Softmax。
        *   **问答 (Question Answering, 如SQuAD):** 需要预测答案在原文中的起始和结束位置，通常会添加两个独立的线性层，分别预测每个token是答案开始位置的概率和是答案结束位置的概率。

3.  **统一的输入格式 (Unified Input Format):**
    *   尽管任务不同，但输入通常需要遵循BERT的标准格式（例如 `[CLS] sentence_A [SEP] sentence_B [SEP]` 或 `[CLS] sentence [SEP]`）。
    *   根据任务需要调整输入的构造方式（例如，对于单句分类，只有`sentence_A`；对于句子对任务，有`sentence_A`和`sentence_B`）。

4.  **联合训练策略 (Joint Training Strategy):**
    *   **数据混合/采样:** 将所有任务的标注数据集合并。在训练时，需要一种策略来从不同任务的数据集中采样样本来构建训练批次（mini-batch）。常见的策略包括：
        *   **均匀采样:** 每个任务被选中的概率相同。
        *   **按比例采样:** 根据每个任务数据集的大小按比例采样。
        *   **轮流采样:** 按顺序或随机顺序轮流为每个任务训练一个或多个批次。
    *   **损失函数计算与聚合:**
        *   对于一个包含来自不同任务样本的批次（或者即使批次只包含一个任务的样本），模型会计算每个样本对应任务的损失（使用其特定的任务头和损失函数，如交叉熵损失、均方误差损失等）。
        *   将来自批次内所有样本（可能跨越多个任务）的损失进行聚合。最简单的方法是**求和或求平均**：`Total_Loss = Loss_Task1 + Loss_Task2 + ...`。
        *   也可以使用**加权求和**：`Total_Loss = w1 * Loss_Task1 + w2 * Loss_Task2 + ...`，其中权重 `w_i` 可以是固定的，也可以是动态调整的（例如，根据任务的难度或学习进度）。
    *   **反向传播与参数更新:**
        *   计算得到的 `Total_Loss` 用于反向传播。
        *   梯度会流经各自的任务头，并**共同更新共享的BERT骨干网络的参数**。这意味着共享的BERT层会同时从所有任务中学习，优化其表示能力以适应所有任务的需求。
        *   各自的任务头参数也只会被对应任务的梯度更新。

**优势:**

*   **参数效率:** 相比为每个任务单独训练一个完整的BERT模型，多任务模型显著减少了所需的总参数量。
*   **知识迁移 (Knowledge Transfer):** 模型可以将在一个任务上学到的知识（例如，语法结构、语义理解）迁移到其他相关任务上，可能提高数据稀疏任务的性能。
*   **正则化效应 (Regularization Effect):** 同时学习多个任务可以作为一种隐式的正则化，有助于防止模型在单个任务上过拟合，提高模型的泛化能力。

#### **大模型时代的微调演进：参数高效微调 (PEFT) 与 LoRA (Low-Rank Adaptation)**

在上面的例子中，BERT-base 只有约 1.1 亿参数，把整个模型的所有参数全部放开一起微调（**全量微调 Full Fine-Tuning**）在一张普通显卡上就能跑通。

但是，当模型演进到今天 70 亿（7B）、700 亿（70B）、6710 亿（671B）甚至 2.8 万亿（2.8T）参数的大模型时，全量微调遇到了巨大的工程瓶颈：不仅训练时需要从上百 GB 到数十 TB 不等的巨量显存来存储梯度与优化器状态，而且每微调一个下游任务，就要保存一份数十 GB 乃至数 TB 的完整模型权重副本。

为了解决这个问题，微软提出的 **[LoRA（Low-Rank Adaptation，低秩自适应，Hu et al., ICLR 2022）](https://arxiv.org/abs/2106.09685)** 成为了当今大模型微调的绝对标配：

![LoRA](https://heidloff.net/assets/img/2023/08/lora.png)

* **核心洞察**：大模型在适应某个具体任务时，权重参数的变化量矩阵 $\Delta W \in \mathbb{R}^{d \times d}$ 其实高度冗余，具有极低的**内在秩（Intrinsic Rank）**。
* **巧妙做法**：**彻底冻结（Freeze）预训练大模型原有的庞大权重 $W_0$ 不动**，仅在旁边并联一高一胖两个极薄的“低秩小矩阵” $B \in \mathbb{R}^{d \times r}$ 和 $A \in \mathbb{R}^{r \times d}$（其中中间瓶颈维度 $r \ll d$，例如原维度 $d=4096$，而秩 $r$ 只需要取 $8$ 或 $16$！）：
  $$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} B \cdot A x$$

**三大实战优势**：
1. **可训练参数暴降 99% 左右**：显存开销锐减，配合量化技术（QLoRA），单张消费级显卡就能微调百亿参数级大模型；
2. **多任务轻量存储与热插拔**：硬盘和显存里只需存一份几十 GB 的基础大模型，不同业务（医疗、法律、客服）各自只需保存和动态切换一个几十到几百 MB 的 LoRA “小插件”；
3. **零额外推理延迟**：单任务上线部署时，可直接用矩阵加法把旁路权重合并回主权重 $W_{\text{new}} = W_0 + \frac{\alpha}{r} B A$，推理速度与原模型一模一样，没有任何额外耗时！

这个实验对真实神经网络在迁移学习（从识别数字 `0-4` 迁移到识别数字 `5-9`）过程中产生的权重变化矩阵 $\Delta W_{\text{full}} = W_{\text{ft}} - W_0 \in \mathbb{R}^{256 \times 256}$（共 65,536 个参数）进行解构，得出以下四项核心结论：

1. **真实梯度下降天然产生“低秩更新”（图 1）：**
   * 对全量微调后的真实权重差值 $\Delta W_{\text{full}}$ 进行奇异值分解（SVD）可以发现，尽管优化器允许全部 256 个正交方向自由更新，但**超过 95% 的更新能量自然坍缩在前 $r=8$ 个奇异值方向上**，第 10 个奇异值之后的 240 多个维度幅值趋近于 0。
   * **背后的数学原因**：线性层的梯度是上游误差信号 $\delta$ 与输入激活向量 $x$ 的外积之和 $\sum \delta_t x_t^T$。由于下游任务的特征流形与分类边界集中在少数几个主方向上，反向传播算出的真实梯度本身就处于一个极低维的子空间中。

2. **后 248 个维度对准确率几乎零贡献（图 2）：**
   * **保留前 $r=8$ 个方向（青色实线）**：不重新训练模型，直接将真实 $\Delta W_{\text{full}}$ 的后 248 个奇异值强制归零，仅用前 8 个奇异值重构权重 $W_0 + \Delta W_{\text{top-8}}$，模型在下游测试集上的准确率与保留全部 256 个奇异值的全量模型**几乎完全一致（约 90%+）**。
   * **保留后 248 个方向（红色点划线）**：反之，若仅抹除前 8 个奇异值、保留其余 **248 个奇异值方向**，模型在测试集上的准确率瞬间暴跌回未微调前的基线水平（约 25%）。这在真实数据上直接证实了全量微调中 96% 以上的参数方向都是冗余的。

3. **真实 LoRA 仅用 6.25% 参数即可复刻全量微调收敛曲线（图 3）：**
   * 在冻结预训练权重 $W_0$、仅训练 $B_{256\times 8}$ 和 $A_{8\times 256}$（4,096 个参数，占全量 65,536 个参数的 **6.25%**）的情况下，LoRA 在全部 6 个 Epoch 中的测试集准确率提升轨迹与全量微调高度重合。

4. **子空间对齐验证：LoRA 自动锁定了全量微调的核心方向（图 4）：**
   * 采用 LoRA 原论文第 7 节的格拉斯曼子空间投影相似度指标 $\phi = \frac{\|U_{\text{full}}^{:k T} U_{\text{LoRA}}^{:r}\|_F^2}{\min(k, r)}$ 进行度量，真实训练出的 LoRA 矩阵 $B \times A$ 与全量微调的前几个主奇异向量具有极高的子空间重合度（远超灰色虚线的随机正交基线）。这说明直接用 $d \times r$ 和 $r \times d$ 的低秩旁路参与梯度下降，能够精准捕捉到全量微调中最核心的任务更新方向。

### 3.5 Prompt Learning / In-Context Learning (ICL)

演化到现在，范式的改革进入深水区了！

#### 3.5.1 Multitask Learning as Question Answering (来源论文：[McCann, et al. 2018](https://arxiv.org/pdf/1806.08730))

核心理念是**万物皆可问答 (Everything as QA)**，理解如何将多种不同的NLP任务统一到问答框架下，并使用单一模型进行多任务学习。<font color="red">**（这可能是我看到的第一个真正意义上尝试出类似大一统学习范式的工作！）**</font>

提出一个 **挑战性的基准 (decaNLP)**，鼓励开发单一模型同时优化多个任务：
- 所有任务的输入都统一为三元组：**(上下文 Context, 问题 Question, 答案 Answer)**。
- 任务的 **类型和目标** 通过自然语言 **问题** 来指定。
- 包含10个公开可用的数据集，涵盖了广泛的NLP任务：
   - **阅读理解 (Question Answering):** SQuAD - 在给定上下文中找答案。
   - **机器翻译 (Machine Translation):** IWSLT (En->De) - 将源语言翻译成目标语言。
   - **文本摘要 (Summarization):** CNN/DailyMail - 为长文档生成摘要。
   - **自然语言推断 (Natural Language Inference):** MNLI - 判断前提和假设之间的关系（蕴含、中性、矛盾）。
   - **情感分析 (Sentiment Analysis):** SST - 判断文本情感（正面/负面）。
   - **语义角色标注 (Semantic Role Labeling):** QA-SRL - 找出句子中谓词的论元（谁对谁做了什么）。
   - **关系抽取 (Relation Extraction):** QA-ZRE - 从文本中抽取指定类型的实体关系（零样本设置）。
   - **目标导向对话 (Goal-Oriented Dialogue):** WOZ - 跟踪对话状态。
   - **语义解析 (Semantic Parsing):** WikiSQL - 将自然语言问题转换为SQL查询。
   - **指代消解 (Pronoun Resolution):** MWSC (Winograd Schema) - 判断代词指向哪个实体。

![decaNLP](https://drive.google.com/uc?id=1uKGEidC1n_GdkUGn7p26y2cDoxbs0Bxo)

> **长远影响:** 这项工作为开发更通用、更强大的 NLP 模型指明了一个有前景的方向，即通过统一的框架（如 QA）和共享的表示来处理多样化的语言任务。它鼓励研究者超越孤立的任务，思考如何构建能够理解和生成语言的通用智能代理。也许最早提出 Prompt Learning 的研究人员也收到了这个工作的启发呢？

#### 3.5.2 Prompt Learning / In-Context Learning (本节图源：[Liu, et al. 2023](https://dl.acm.org/doi/pdf/10.1145/3560815))

随着模型规模的急剧增大（如 GPT-3），研究者发现这些超大型语言模型 (LLMs) 展现出一种惊人的 **上下文学习 (In-Context Learning, ICL)** 能力。无需进行模型参数的更新（即无需微调），仅仅通过在 **输入提示 (Prompt)** 中给出任务描述或若干示例 (demonstrations)，模型就能理解任务要求并给出正确的输出。
- **Zero-shot Learning:** 只给出任务描述，不给示例。
- **Few-shot Learning:** 在提示中给出少量（通常 1 到几十个）任务示例。

![prompts.png](https://drive.google.com/uc?id=1OT7Lcu12Uq9pWLjH1f3RPwure05a0_hv)

**Prompt Engineering** 成为了与这些大型模型交互的关键技术，即如何设计有效的提示来引导模型完成特定任务。

微调 + 预训练 (Pre-training/Fine-tuning) 虽然减少了标注数据需求，但仍需要为每个任务进行微调，管理多个微调后的模型副本。对于新任务，仍需要一定的准备时间。相比之下，Prompt Learning 具备以下优势：
1.  **无需微调:** 对于很多任务，可以直接使用预训练好的大模型，通过 Prompt 实现任务适配，极大提高了灵活性和效率。
2.  **任务大一统的进一步体现:** 一个单一的、巨大的预训练模型，通过不同的 Prompt，可以执行成百上千种不同的任务，真正实现了“一个模型打天下”的潜力。
3.  **交互方式的变革:** 从“训练模型适应任务”转向“设计提示引导模型”。

<img src="https://drive.google.com/uc?id=1zyzzvqPWHeF3k74JJA45r6PZerU6vlL4" width=800px>

> 在 2021 年前后，Prompt Learning（提示学习）刚露头时，确实被学术界和工业界疯狂吐槽。当时大家管它叫 **“炼丹术”** 或 **“玄学”**，因为早期模型的 Prompt 极度脆弱：
> *   你用“*请把以下句子翻译成英文：*”能成功；
> *   你换成“*翻译下面的句子：*”（少了个‘请’和‘英文’），模型可能就直接罢工或者开始胡说八道。
> *   当时甚至有专门的论文去研究：为什么多一个空格、少一个逗号，模型的性能会断崖式下跌 30%。
>
> 那么，这个被视为“玄学”的玩具，后来是如何一步步逆袭，成为今天统治整个 AI 界的“大一统范式”的？

#### 3.5.3 指令微调 (Instruction Tuning) 与 对齐 (Alignment)

即使是大型的预训练模型，有时也难以完全遵循用户的复杂指令或生成符合人类期望（如有帮助、诚实、无害）的回复。
- **指令微调 (Instruction Tuning):** 在预训练之后，使用大量由 **(指令, 输出)** 对组成的数据集对模型进行微调。这些指令涵盖了各种各样的任务。这使得模型能更好地理解和遵循自然语言指令，提高其零样本和少样本任务的泛化能力（如 FLAN, T0, InstructGPT）。
- **对齐 (Alignment):** 确保模型的行为符合人类的意图和价值观。一个关键技术是 **基于人类反馈的强化学习 (Reinforcement Learning from Human Feedback, RLHF)。** 大致流程是：
  1.  收集人类对模型不同输出的偏好排序数据。
  2.  用这些数据训练一个“奖励模型 (Reward Model)”，使其能够评估模型输出的质量（符合人类偏好的程度）。
  3.  使用强化学习算法（如 PPO），根据奖励模型的信号来微调 LLM，使其倾向于生成更高奖励（即更符合人类偏好）的输出。

这些方法，缓解了纯 Prompt/ICL 模型可能不理解指令、产生不相关或有害内容、难以控制的问题，使得模型具备：
1.  **更好的指令遵循能力:** Instruction Tuning 使模型更像一个“指令执行器”。
2.  **提高有用性、诚实性和无害性:** RLHF 等对齐技术使模型输出更符合人类期望和社会规范。
3.  **更强的对话和交互能力:** 这些技术是 ChatGPT 等先进对话 AI 的关键。

### 3.6 后训练慢思考范式：可验证强化学习 (RLVR) 与测试时计算扩展 (2024～)

前面我们讲到了通过指令微调（SFT）和基于人类反馈的强化学习（RLHF）训练出了对答如流的 ChatGPT。然而到了 2024 年，这套范式撞上了两堵高墙：
1. **预训练数据墙（Data Wall）**：全人类互联网上的高质量公开文本几乎被大模型吃光了，单纯靠堆数据、堆参数的预训练扩展定律（Pre-training Scaling Law）边际效益递减。
2. **“快思考（System 1）”的推理瓶颈**：传统自回归模型无论回答“1+1等于几”还是证明“黎曼猜想”，生成每个词消耗的算力几乎是一样的——它就像一个不许打草稿、必须脱口而出的考生，一旦前两步直觉走偏，后面就会一路圆谎错到底。

2024 年底至 2025 年初，随着 **OpenAI o1 / o3** 与 **[DeepSeek-R1](https://arxiv.org/abs/2501.12948)** 的问世，大模型的训练与使用范式迎来了新的革命：**后训练可验证强化学习（RLVR）与测试时计算扩展（Test-Time Compute Scaling）**！

#### **从“训练时堆算力”到“测试时多思考（Test-Time Scaling）”**
心理学家卡尼曼在《思考，快与慢》中将人类思维分为直觉式的快思考（System 1）与深思熟虑的慢思考（System 2）。
新一代推理模型在给出最终回答前，会先在 `<think> ... </think>` 标签内生成几千甚至上万字的内部思维链（长草稿）：它会在草稿里自己列方程、试错、发现矛盾、推翻重来。研究者发现了一条全新的扩展定律——只要让模型在回答难题时多“想”一会儿（增加测试推理阶段生成的思考 Token 数量），模型的数学、编程和复杂逻辑推理准确率就会成倍跃升！

#### **DeepSeek-R1 是如何练成“慢思考”的？**
过去大家以为，要教模型学会长链推理，必须找人类数学家手写几万条解题步骤去喂给模型（SFT），或者训练一个复杂的神经网络给每一步打分（过程奖励模型 PRM）。但 **[DeepSeek-R1 论文](https://arxiv.org/abs/2501.12948)** 证明了一条极其纯粹的道路：

* **基于可验证奖励的强化学习（RLVR, RL with Verifiable Rewards）**：
  对于数学、代码、逻辑推理题，完全不需要人类标注员或神经网络裁判，直接用 100% 客观的规则验证器（Rule-based Verifier）打分——数学题最终答案算对了给 `+1` 分，代码跑通了沙箱单元测试给 `+1` 分，思考过程乖乖写在 `<think></think>` 格式里给格式分！
* **惊人的“顿悟时刻（Aha Moment）”**：
  在 DeepSeek-R1-Zero 实验中，研发团队甚至完全跳过了监督微调（SFT），直接把基础模型扔进纯 RLVR 环境里自己试错。训练到几千步时，模型为了拿到答对题目的奖励，竟然无师自通地涌现出了“自我反思”能力——它会在思维链里自己突然喊停：*“Wait, wait. Wait. That's an aha moment I can flag here... Let's re-evaluate this step-by-step!（等等，这里不对，让我重新检查一遍！）”*
* **省下一半显存的强化学习神器：[GRPO（群组相对策略优化，DeepSeekMath/R1）](https://arxiv.org/abs/2402.03300)**：
  传统的 PPO 强化学习算法需要额外构建一个和主模型一样庞大的“评论家网络（Critic Model）”来预估基准分，显存开销翻倍。**GRPO 彻底砍掉了 Critic 网络**：针对同一道题目 $q$，直接让模型自己采样生成一组 $G$ 个不同回答 $\{o_1, o_2, \dots, o_G\}$，用规则判卷得到一组分数 $\{r_1, r_2, \dots, r_G\}$，然后直接用**这一组回答自己的平均分 $\mu_r$ 和标准差 $\sigma_r$** 来算谁表现好、谁表现差：
  $$\hat{A}_i = \frac{r_i - \text{mean}(r_1, \dots, r_G)}{\text{std}(r_1, \dots, r_G) + \epsilon}$$
  高于组内平均分的思考方式就奖励强化，低于平均分的就抑制——简单、优雅、极其高效！

### ✅ 小结：范式的演进回顾

|年代 | 阶段 | 方法 | 主要特点 |
|---------|------|-----------|----------|
| ～ | 规则系统	| if-else/正则	| 脆弱、可解释
| 1970s～1990s | 统计学习	| SVM/NB/HMM	| 特征依赖专家
| 1980s～2000s | 深度学习	| CNN/RNN	| 表达力强、但重任务
| 2018 | 预训练 + 微调	| BERT	| 统一表示，任务分离
| 2021 | Prompt Learning	| GPT	| 自然语言接口，统一任务
| 2024～ | 后训练强化与慢思考 (RLVR / TTCS) | OpenAI o1/o3, DeepSeek-R1 (GRPO) | 可验证奖励激发内生反思推理链，用思考时间换智能上限

> 🌟 NLP 的范式经历了从 <strong><font color="red">规则驱动</font></strong> 到 <strong><font color="red">统计学习</font></strong>，再到 <strong><font color="red">任务特定的深度学习</font></strong>，最终演变为 <strong><font color="red">预训练-微调</font></strong> 的主导模式。随着模型规模的增长，<strong><font color="red">Prompting / In-Context Learning</font></strong> 成为与超大型模型交互的新方式，进一步推动了任务的大一统。随后，<strong><font color="red">Instruction Tuning</font></strong> 和 <strong><font color="red">Alignment (如 RLHF)</font></strong> 正在使这些强大的模型更加可控、有用和安全。这一系列范式的革命，使得开发和应用 NLP 技术的方式发生了根本性的变化。

<div class="alert alert-success">
🌟 通过以上三个角度的回顾，我们可以看到：<br>
   ☑️ <strong><font color="red">表示的演进</font></strong> 提供了更丰富、更动态地捕捉信息含义的基础。<br>
   ☑️ <strong><font color="red">架构的演进</font></strong> 提供了更强大、更高效地处理信息（尤其是序列信息）的引擎。<br>
   ☑️ <strong><font color="red">范式的演进</font></strong> 则改变了我们构建、训练和使用模型的方式，实现了前所未有的通用性和易用性。<br>
    
    
关键转变特征：
1. 人工编码 → 数据驱动
2. 专用系统 → 通用能力
3. 明确指令 → 自然交互
4. 单一模态 → 多模态融合
5. 被动响应 → 主动规划

    
这三大脉络相互交织、相互促进，共同推动了自然语言处理领域的发展，最终造就了今天我们所见到的强大而通用的 LLM。理解这段历史，有助于我们更好地认识 LLM 的能力边界、内在机理，并思考其未来的发展方向。
</div>